<a href="https://colab.research.google.com/github/BalytskyiJaroslaw/BioMedAgent_Adapted_for_Google_Colab/blob/main/BioMedAgent_adapted_for_Google_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title **🔧 Set Up the Environment in Colab** { display-mode: "form" }

import os
import sys
import shutil
import subprocess

def run_cmd(cmd, check=True):
    print(">>", " ".join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed: {' '.join(cmd)}")
    return result

print("🦁 Upgrading base Python tooling...")
run_cmd([sys.executable, "-m", "pip", "install", "--upgrade", "pip", "setuptools", "wheel"])

print("\n🦁 Installing Python packages...")
packages = [
    "colorama",
    "redis==5.0.3",
    "BCEmbedding==0.1.5",
    "transformers==4.36.2",
    "tokenizers==0.15.2",
    "huggingface-hub==0.36.2",
    "requests==2.32.4",
    "sentence-transformers==3.0.1",
    "scikit-learn",
    "pandas",
    "matplotlib",
    "tqdm",
]
run_cmd([sys.executable, "-m", "pip", "install", *packages])

print("\n🦁 Checking for apt-get / redis-server...")
apt_exists = shutil.which("apt-get") is not None
redis_server_exists = shutil.which("redis-server") is not None
redis_cli_exists = shutil.which("redis-cli") is not None

if apt_exists and not redis_server_exists:
    print("🦁 Installing redis-server via apt...")
    run_cmd(["apt-get", "update", "-qq"])
    run_cmd(["apt-get", "install", "-y", "redis-server"])
    redis_server_exists = shutil.which("redis-server") is not None
    redis_cli_exists = shutil.which("redis-cli") is not None
elif not apt_exists:
    print("⚠️ apt-get is not available in this environment.")
else:
    print("✅ redis-server already present.")

if redis_server_exists:
    print("\n🦁 Starting Redis...")
    run_cmd(["redis-server", "--daemonize", "yes"], check=False)

    if redis_cli_exists:
        ping = run_cmd(["redis-cli", "ping"], check=False)
        if "PONG" in ping.stdout:
            print("✅ Redis is running.")
        else:
            print("⚠️ Redis did not respond with PONG.")
    else:
        print("⚠️ redis-cli not found, so ping check was skipped.")
else:
    print("⚠️ redis-server is still unavailable. Install it manually if your workflow needs it.")

print("\n🎉 Base setup complete.")

🦁 Upgrading base Python tooling...
>> /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.0 MB/s eta 0:00:00
  Attempting uninstall: wheel
    Found existing installation: wheel 0.46.3
    Uninstalling wheel-0.46.3:
      Successfully uninstalled wheel-0.46.3
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.

🦁 Installing Python packages...
>> /usr/bin/

In [2]:
# @title 🦁 Choose a task and upload the required input file(s) 📂 { display-mode: "form" }
#@markdown ### Demo data examples
#@markdown - Go [here](https://github.com/BalytskyiJaroslaw/BioMedAgent_Adapted_for_Google_Colab/tree/main/Demo_data) to download the demo data examples (**Demo_data** folder).
#@markdown - Example outputs are [here](https://github.com/BalytskyiJaroslaw/BioMedAgent_Adapted_for_Google_Colab/tree/main/Demo_outputs) to download the demo data examples (**Demo_outputs folder**).

import os
import shutil
import requests
from getpass import getpass
from google.colab import files

# ------------------------------------------------------------
# 1) hidden API key input
# ------------------------------------------------------------
api_key = getpass("Enter your OpenAI API key (input hidden): ").strip()
if not api_key:
    raise ValueError("No API key was entered.")

os.environ["OPENAI_API_KEY"] = api_key
os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1"

print("\n✅ API key stored in environment.")
print("OPENAI_BASE_URL =", os.environ["OPENAI_BASE_URL"])

# Quick API test
headers = {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}
r = requests.get("https://api.openai.com/v1/models", headers=headers)

print("\n🔎 API test")
print("STATUS:", r.status_code)
print("TEXT:", r.text[:200])

if r.status_code != 200:
    raise RuntimeError("API key check failed. Please verify your key.")

# ------------------------------------------------------------
# 2) choose task type from Colab dropdown
# ------------------------------------------------------------
TASK_TYPE = "statistics_t_test"  # @param ["statistics_t_test", "machine_learning", "statistics_qq_plot", "visualization_survival_plot", "visualization_violin_plot"]

print("\n==============================")
print("Selected TASK_TYPE =", TASK_TYPE)
print("==============================")

# ------------------------------------------------------------
# 3) local upload folder
# ------------------------------------------------------------
UPLOAD_DIR = "/content/BioMedAgent_UserData"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# ------------------------------------------------------------
# 4) initialize all path variables
# ------------------------------------------------------------
DATA1_PATH = None
GROUP1_PATH = None
HEART_PATH = None
BOXPLOT_PATH = None
SURVIVAL_PATH = None
PLOT_PATH = None

# ------------------------------------------------------------
# 5) task -> required files + user guidance
# ------------------------------------------------------------
task_requirements = {
    "statistics_t_test": {
        "files": [
            ("data1.tsv", "Biomarker table: rows = biomarkers, columns = sample names"),
            ("group1.tsv", "Grouping table: columns must include ID and group; groups should be 1 and 2"),
        ],
        "message": (
            "You selected statistics_t_test.\n"
            "Please upload these TWO files in this order:\n"
            "  1) data1.tsv  -> biomarker data\n"
            "  2) group1.tsv -> grouping info with columns ID and group"
        )
    },
    "machine_learning": {
        "files": [
            ("heart_disease.csv", 'Dataset where "Target" is the target column'),
        ],
        "message": (
            "You selected machine_learning.\n"
            "Please upload:\n"
            '  1) heart_disease.csv -> dataset containing the "Target" column'
        )
    },
    "statistics_qq_plot": {
        "files": [
            ("boxplot.tsv", "Dataset containing columns Group2 and Value"),
        ],
        "message": (
            "You selected statistics_qq_plot.\n"
            "Please upload:\n"
            "  1) boxplot.tsv -> must contain Group2 and Value columns"
        )
    },
    "visualization_survival_plot": {
        "files": [
            ("TCGA_LIHC_survival.txt", "Survival table containing OS.time, OS, and gender"),
        ],
        "message": (
            "You selected visualization_survival_plot.\n"
            "Please upload:\n"
            "  1) TCGA_LIHC_survival.txt -> must contain OS.time, OS, and gender columns"
        )
    },
    "visualization_violin_plot": {
        "files": [
            ("plot.tsv", "Dataset containing Sex, Age_Group, and WBC columns"),
        ],
        "message": (
            "You selected visualization_violin_plot.\n"
            "Please upload:\n"
            "  1) plot.tsv -> must contain Sex, Age_Group, and WBC columns"
        )
    },
}

if TASK_TYPE not in task_requirements:
    raise ValueError(f"Unsupported TASK_TYPE: {TASK_TYPE}")

print(task_requirements[TASK_TYPE]["message"])
print()

# ------------------------------------------------------------
# 6) helper for one-file upload
# ------------------------------------------------------------
def upload_one_file(expected_name, description):
    print(f"📄 Please upload: {expected_name}")
    print(f"   Description: {description}")

    uploaded = files.upload()

    if len(uploaded) != 1:
        raise ValueError(f"Please upload exactly ONE file for {expected_name}")

    uploaded_name = list(uploaded.keys())[0]
    tmp_path = os.path.join("/content", uploaded_name)
    final_path = os.path.join(UPLOAD_DIR, expected_name)

    # remove old file if present
    if os.path.exists(final_path):
        os.remove(final_path)

    if os.path.abspath(tmp_path) != os.path.abspath(final_path):
        shutil.move(tmp_path, final_path)

    print(f"✅ Saved as: {final_path}\n")
    return final_path

# ------------------------------------------------------------
# 7) upload only the files required for the selected task
# ------------------------------------------------------------
uploaded_paths = {}

for expected_name, description in task_requirements[TASK_TYPE]["files"]:
    uploaded_paths[expected_name] = upload_one_file(expected_name, description)

# ------------------------------------------------------------
# 8) assign the canonical path variables
# ------------------------------------------------------------
if "data1.tsv" in uploaded_paths:
    DATA1_PATH = uploaded_paths["data1.tsv"]

if "group1.tsv" in uploaded_paths:
    GROUP1_PATH = uploaded_paths["group1.tsv"]

if "heart_disease.csv" in uploaded_paths:
    HEART_PATH = uploaded_paths["heart_disease.csv"]

if "boxplot.tsv" in uploaded_paths:
    BOXPLOT_PATH = uploaded_paths["boxplot.tsv"]

if "TCGA_LIHC_survival.txt" in uploaded_paths:
    SURVIVAL_PATH = uploaded_paths["TCGA_LIHC_survival.txt"]

if "plot.tsv" in uploaded_paths:
    PLOT_PATH = uploaded_paths["plot.tsv"]

# ------------------------------------------------------------
# 9) verify selected-task files
# ------------------------------------------------------------
print("📂 Final file check")
for fname, fpath in uploaded_paths.items():
    print(f"{fname}: exists? {os.path.exists(fpath)} -> {fpath}")
    if not os.path.exists(fpath):
        raise FileNotFoundError(f"Missing file after upload: {fpath}")

print("\n🎉 Upload step complete.")

# ------------------------------------------------------------
# 10) build question_info exactly like demo.py, but with local paths
# ------------------------------------------------------------
print("\nTASK_TYPE =", TASK_TYPE)
if TASK_TYPE == "statistics_t_test":
    print("data1 exists?", os.path.exists(DATA1_PATH))
    print("group1 exists?", os.path.exists(GROUP1_PATH))

base_info = {
    "machine_learning": {
        "question": 'I have a dataset {heart_disease.csv}, the "Target" column is the target column and the other columns are the feature columns, please perform a singular value decomposition downscaling on this dataset.',
        "files": [{
            "name": "heart_disease.csv",
            "path": HEART_PATH
        }]
    },
    "statistics_t_test": {
        "question": "There are two existing data tables, {data1.tsv} is a biomarker data table, with sample names as column names and biomarker names as row names, {group1.tsv} is a data table containing sample grouping information, where the two groups are 1 and 2, with the column name ID as sample names and the column name group as grouping information. Please use the tool t_test to perform an independent sample t-test based on this information.",
        "files": [
            {"name": "data1.tsv", "path": DATA1_PATH},
            {"name": "group1.tsv", "path": GROUP1_PATH},
        ]
    },
    "statistics_qq_plot": {
        "question": "There is a dataset {boxplot.tsv}, Group2 column is the grouping information, which contains treat1 and treat2, please analyze whether the data in Value column of the two groupings of treat1 and treat2 have similar distribution characteristics, and plot QQ plot.",
        "files": [{
            "name": "boxplot.tsv",
            "path": BOXPLOT_PATH
        }]
    },
    "visualization_survival_plot": {
        "question": "There is a data table {TCGA_LIHC_survival.txt} that contains information related to patient survival. The column named OS.time represents the survival time of the patients, the column named OS indicates the survival event status of the patients, and the column named gender is a variable that affects survival. Please use the tool survival_curve to plot the survival curve based on this information.",
        "files": [{
            "name": "TCGA_LIHC_survival.txt",
            "path": SURVIVAL_PATH
        }]
    },
    "visualization_violin_plot": {
        "question": "There is a data file {plot.tsv}, the Sex column is the gender of the sample, Age_Group is the different age grouping corresponding to each gender, please use the violin plot to show the distribution of WBC for each age grouping of different genders.",
        "files": [{
            "name": "plot.tsv",
            "path": PLOT_PATH
        }]
    },
}

question_info = base_info[TASK_TYPE]

print("\n✅ question_info is ready for Cell 27.")
print("question_info =", question_info)

Enter your OpenAI API key (input hidden): ··········

✅ API key stored in environment.
OPENAI_BASE_URL = https://api.openai.com/v1

🔎 API test
STATUS: 200
TEXT: {
  "object": "list",
  "data": [
    {
      "id": "gpt-4-0613",
      "object": "model",
      "created": 1686588896,
      "owned_by": "openai"
    },
    {
      "id": "gpt-4",
      "object": "mo

Selected TASK_TYPE = visualization_survival_plot
You selected visualization_survival_plot.
Please upload:
  1) TCGA_LIHC_survival.txt -> must contain OS.time, OS, and gender columns

📄 Please upload: TCGA_LIHC_survival.txt
   Description: Survival table containing OS.time, OS, and gender


Saving TCGA_LIHC_survival.txt to TCGA_LIHC_survival.txt
✅ Saved as: /content/BioMedAgent_UserData/TCGA_LIHC_survival.txt

📂 Final file check
TCGA_LIHC_survival.txt: exists? True -> /content/BioMedAgent_UserData/TCGA_LIHC_survival.txt

🎉 Upload step complete.

TASK_TYPE = visualization_survival_plot

✅ question_info is ready for Cell 27.
question_info = {'question': 'There is a data table {TCGA_LIHC_survival.txt} that contains information related to patient survival. The column named OS.time represents the survival time of the patients, the column named OS indicates the survival event status of the patients, and the column named gender is a variable that affects survival. Please use the tool survival_curve to plot the survival curve based on this information.', 'files': [{'name': 'TCGA_LIHC_survival.txt', 'path': '/content/BioMedAgent_UserData/TCGA_LIHC_survival.txt'}]}


In [8]:
# @title 🦁⚙️ **BioMedAgent** model initialization { display-mode: "form" }

import os

os.environ["OPENAI_BASE_URL"] = "https://api.openai.com/v1/chat/completions"


# Cell number 3
# config.py

import os
import redis
from datetime import datetime

def time_dir():
    now = datetime.now()
    return os.path.join(f"{now.year}-{now.month}", f"{now.day}")

def _get_base_dir():
    # If running from a real .py file
    if "__file__" in globals():
        return os.path.abspath(os.path.dirname(__file__))
    # If running inside Jupyter/Colab
    return "/content/BioMedAgent"

class Config:
    LLM_CALL_PASSWORD = "BioMedAgent"
    LLM_CALL_WAITING_TIME = 1
    LLM_SERVER_IP = ""

    BASE_DIR = _get_base_dir()
    LOG_DIR = os.path.join(BASE_DIR, "log")
    TASK_DIR = os.path.join(BASE_DIR, "task")
    TOOL_DOC_DIR = os.path.join(BASE_DIR, "tool", "doc")
    TOOL_CODE_DIR = os.path.join(BASE_DIR, "tool", "code")
    ZIP_DIR = os.path.join(BASE_DIR, "zip")

    SAVE_LOG = True
    ECHO_INFO = True

    SAVE_MEMORY = False
    MEMORY_PREFIX = "v1.0.0"

    USE_MEMORY = False
    USE_FILE_APPENDIX = False

    BASE_LLM_MODEL = "gpt-4o-mini"
    SUPER_LLM_MODEL = "gpt-4o-mini"

    HIGHSCORE_TOOL_THRESHOLD = 5
    WORKFLOW_USED_TOOL_THRESHOLD = 5

    EXECUTOR_CODE_WAITING_TIME = 1
    ACTION_RETRY_TIMES = 4

    REDIS_HOST = "localhost"
    REDIS_PORT = 6379

    REDIS_GPT_TASK_KEY = "biomedagent:gpt:task"
    REDIS_GPT_RESULT_KEY = "biomedagent:gpt:result"

    REDIS_LLAMA_TASK_KEY = "biomedagent:llama:task"
    REDIS_LLAMA_RESULT_KEY = "biomedagent:llama:result"

    REDIS_EXECUTOR_LIST_TASK_KEY = "biomedagent:executor:task"
    REDIS_EXECUTOR_LIST_DATA_KEY = "biomedagent:executor:data"

    REDIS_ACTIVE_TOOL_KEY = "biomedagent:tools:active"
    REDIS_TOOL_INFO_KEY = "biomedagent:tools:info"
    REDIS_STATUS_DATA_KEY = "biomedagent:status:data"
    REDIS_STATUS_LIST_KEY = "biomedagent:status:list"
    REDIS_PROGRESS_DATA_KEY = "biomedagent:progress_data"
    REDIS_RESULT_DATA_KEY = "biomedagent:result_data"

    REDIS_INSTANCE_INFO_KEY = "biomedagent:instance:info"
    REDIS_INSTANCE_PUBLISH_KEY = "biomedagent:instance:publish"

    REDIS_MEMORY_STORAGE_KEY = f"biomedagent:memory:{MEMORY_PREFIX}:storage"
    REDIS_MEMORY_INFO_KEY = "biomedagent:memory:info"
    REDIS_MEMORY_TASK_KEY = f"biomedagent:memory:{MEMORY_PREFIX}:task"
    REDIS_MEMORY_RESULT_KEY = f"biomedagent:memory:{MEMORY_PREFIX}:result"
    REDIS_MEMORY_LOG = f"biomedagent:memory:{MEMORY_PREFIX}:log:"
    REDIS_MEMORY_LOG2 = f"biomedagent:memory:{MEMORY_PREFIX}:log"

    def __init__(self, task_id=None):
        self.task_id = task_id
        self.time_path = time_dir()

    def get_task(self):
        if self.task_id is None:
            raise ValueError("task id is None")
        return f"{self.task_id}"

    def get_redis_connect(self):
        return redis.Redis(
            host=self.REDIS_HOST,
            port=self.REDIS_PORT,
            decode_responses=True
        )

    @classmethod
    def set_task(cls, task_id):
        return cls(task_id)

    def log_error(self, item: str, message: str):
        time_now = datetime.now()
        log_dir = os.path.join(self.BASE_DIR, "BioLog")
        os.makedirs(log_dir, exist_ok=True)
        with open(os.path.join(log_dir, "error.log"), "a", encoding="utf8") as f:
            f.write(f"item={item}\ntime={time_now}\nmessage={message}\n{'='*50}\n")

config = Config()

print("BASE_DIR =", config.BASE_DIR)
print("LOG_DIR  =", config.LOG_DIR)

# Cell number 4

# utils.py

import os
import json
from datetime import datetime
from uuid import uuid4

from colorama import Fore, Style
#from config import Config

rprint = lambda x: print(Fore.RED + x + Style.RESET_ALL)
gprint = lambda x: print(Fore.GREEN + x + Style.RESET_ALL)
yprint = lambda x: print(Fore.YELLOW + x + Style.RESET_ALL)

def time_dir():
    now = datetime.now()
    return os.path.join(
        f"{now.year}-{now.month}",f"{now.day}"
    )

def generate_task_id():
    return f"{uuid4()}"

def get_action_code(programmer_code, tester_code):
    total_code = f"""

{programmer_code}

{tester_code}
"""
    return total_code


def knowledge_base(item_key:str, query_key:str, query_value:str, question_id:str, task_id = None):

    r = Config().get_redis_connect()
    memory_id = generate_task_id()
    r.lpush(
        f"{Config.REDIS_MEMORY_STORAGE_KEY}:{item_key}", json.dumps({
            "key":query_key,
            "value":query_value,
            "id":memory_id,
            "question_id":question_id,
            "task_id":task_id,
            "item_key":item_key
        })
    )
    r.hset(Config.REDIS_INSTANCE_INFO_KEY, memory_id, json.dumps({
        "key":query_key,
        "value":query_value,
        "id":memory_id,
        "question_id":question_id,
        "task_id":task_id,
        "item_key":item_key
    }))


import ast

def extract_function(code_str, function_name):
    tree = ast.parse(code_str)

    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and node.name == function_name:
            return ast.unparse(node)

    return None

# Cell number 5

# scripts -> llm.py

import asyncio
import os
import json
import requests
import time
from uuid import uuid4

# Assumes Config is already defined in another notebook cell

# notebook-safe fallbacks
def gprint(x):
    print(x)

def generate_task_id():
    return str(uuid4())

def handle_text(func):
    def inner(uid, config: Config):
        ok, msg = func(uid, config)

        if config.ECHO_INFO and ok:
            print(msg)
            gprint("=" * 30)

        if (not config.SAVE_LOG) or (not ok):
            return ok, msg

        task_id = config.get_task()
        log_path = os.path.join(config.LOG_DIR, config.time_path)
        os.makedirs(log_path, exist_ok=True)

        with open(os.path.join(log_path, f"{task_id}.log"), "a", encoding="utf8") as f:
            f.write(f"{msg}\n{'='*30}\n")

        return ok, msg

    async def wait(uid, config: Config):
        ok = False
        msg = ""
        while not ok:
            ok, msg = inner(uid, config)
            if not ok:
                await asyncio.sleep(config.LLM_CALL_WAITING_TIME)
        return msg

    return wait

def llmcall(model, temperature, messages, config: Config):
    uid = generate_task_id()
    data = {
        "task_id": uid,
        "messages": messages,
        "temperature": temperature,
        "model": model,
    }

    r = config.get_redis_connect()
    r.lpush(config.REDIS_GPT_TASK_KEY, json.dumps(data))
    return uid

@handle_text
def llmresponse(uid: str, config: Config):
    r = config.get_redis_connect()
    result = r.hget(config.REDIS_GPT_RESULT_KEY, uid)

    if not result:
        return False, ""

    result = json.loads(result)
    return True, result["response"]

async def demo_llm_call():
    task_id = str(uuid4())
    config = Config.set_task(task_id)

    uid = llmcall(
        model="gpt-4-turbo",
        temperature=0.5,
        messages=[
            {
                "role": "system",
                "content": "you are a helpful ai"
            },
            {
                "role": "user",
                "content": "你好啊"
            }
        ],
        config=config
    )

    msg = await llmresponse(uid, config)
    print("Final response:", msg)
    return msg

# Cell number 6

# scripts->chat.py

from copy import deepcopy
from uuid import uuid4
import asyncio

# Assumes Config, llmcall, llmresponse are already defined in previous notebook cells

class ChatSession:
    def __init__(self, model, temperature, config, messages=None) -> None:
        self.model = model
        self.temperature = temperature
        self.config = config
        self.messages = [] if messages is None else deepcopy(messages)
        self.__lock = False

    def copy(self):
        return ChatSession(
            self.model,
            self.temperature,
            self.config,
            self.messages
        )

    def set_system(self, system):
        self.messages.append({
            "role": "system",
            "content": system
        })

    async def chat(self, content):
        if self.__lock:
            raise RuntimeError("This ChatSession is already processing a message.")

        self.__lock = True
        try:
            self.messages.append({
                "role": "user",
                "content": content
            })

            uid = llmcall(
                self.model,
                self.temperature,
                self.messages,
                self.config
            )

            msg = await llmresponse(uid, self.config)

            self.messages.append({
                "role": "assistant",
                "content": msg
            })

            return msg
        finally:
            self.__lock = False


async def demo_single_chat():
    task_id = str(uuid4())
    config = Config.set_task(task_id)

    session = ChatSession(
        model="gpt-3.5-turbo",
        temperature=0.5,
        config=config
    )

    session.set_system("ai")

    reply = await session.chat("今天怎么样")
    print(reply)
    return reply


async def demo_batch_chat():
    import random

    task_id = str(uuid4())
    config = Config.set_task(task_id)

    questions = [
        f"{a} {b} {c} = ?"
        for a, b, c in zip(
            random.choices(range(1, 100), k=5),
            random.choices("+-*/", k=5),
            random.choices(range(1, 100), k=5)
        )
    ]

    tasks = []
    for question in questions:
        session = ChatSession(
            model="gpt-3.5-turbo",
            temperature=0.5,
            config=config
        )
        session.set_system("ai")
        tasks.append(asyncio.create_task(session.chat(question)))

    results = await asyncio.gather(*tasks)

    for q, r in zip(questions, results):
        print("Q:", q)
        print("A:", r)
        print("-" * 40)

    return questions, results

# Cell number 7

# scripts -> retriever.py

import os
import re
import json

from BCEmbedding import RerankerModel

# Assumes Config is already defined in another notebook cell

class InstanceRetriever:
    def __init__(self, model_path=None) -> None:
        self.r = Config().get_redis_connect()

        # Default to repo-local model folder
        if model_path is None:
            model_path = os.path.join(Config.BASE_DIR, "model", "bce-reranker-base_v1")

        self.model_path = model_path
        self.reranker_model = RerankerModel(model_name_or_path=self.model_path)
        self._instance_dict = self._prepare()

    def _get_instances_info(self, instance_id: str):
        info = self.r.hget(Config.REDIS_INSTANCE_INFO_KEY, instance_id)
        if not info:
            raise Exception("非法instance_id")

        info = json.loads(info)

        result = self.r.get(f"biomedagent:result:{instance_id}")
        files = None

        if result:
            batch_info = self.r.get(f"biomedagent:info:batch_task:{instance_id}")
            normal_info = self.r.get(f"biomedagent:info:normal_task:{instance_id}")

            if batch_info:
                batch_info = json.loads(batch_info)
                files = batch_info.get("files")
            elif normal_info:
                normal_info = json.loads(normal_info)
                files = normal_info.get("files")

        question = info.get("question")
        if not question:
            raise Exception("非法instance数据")

        if not files:
            files = info.get("files")

        if not files:
            files = re.findall(r"\{(.*?)\}", question)

        return question, files

    def _prepare(self):
        instance_dict = {}
        instances = self.r.hkeys(Config.REDIS_INSTANCE_PUBLISH_KEY)

        for instance_id in instances:
            question, files = self._get_instances_info(instance_id)
            publish_raw = self.r.hget(Config.REDIS_INSTANCE_PUBLISH_KEY, instance_id)
            publish_info = json.loads(publish_raw)

            instance_dict[question] = {
                "question": question,
                "files": files,
                "id": instance_id,
                "type": publish_info["type"]
            }

        return instance_dict

    def match(self, prompt, k=5):
        self._instance_dict = self._prepare()

        if prompt == "":
            prompt = "我"

        instances = list(self._instance_dict.keys())
        if not instances:
            return []

        rerank_results = self.reranker_model.rerank(prompt, instances)
        top_k_instances = rerank_results["rerank_passages"][:k]

        result = []
        for item in top_k_instances:
            result.append({
                "question": item,
                "id": self._instance_dict[item]["id"],
                "files": self._instance_dict[item]["files"],
                "type": self._instance_dict[item]["type"]
            })

        return result

# Cell number 8

# scripts -> prompt.py

linguist_system = """
You are a linguist and you can determine what language the user's prompt is in, you have to determine if the user's prompt is stated in English and give the result of your judgment. Your response should follow the following form: use <RESULT></RESULT> tag to wrap the result of your judgment, and do not respond with anything other than the <RESULT> tag. You can only reply with <RESULT>YES</RESULT> or <RESULT>NO</RESULT>.
"""

linguist_prompt = """
Now that the user's prompt is {{prompt}}, give your judgment.
"""

translator_system = """
You are a translator with an excellent biomedical background who can accurately translate the prompt given by the user into English without any embellishments and must accurately translate every word of the user into English. Your reply should follow the following format: wrap your translated prompt with <PROMPT></PROMPT> tag and don't reply with anything else but the prompt, eg: <PROMPT>new prompt</PROMPT>.Translate only the user's prompt, do not add other content.
"""

translator_prompt = """
{{prompt}}
"""

prompt_engineer_system = """
You are a good careful and cautious prompt engineer with biomedical background,
your job is to carefully analyze the user's needs based on the query prompt given by the user and to expand and refine the user's prompt by completing the ambiguous semantics in the user's prompt and filling in the appropriate details,
and building the prompt in points, for example: Tasks, Files, Requirements, Points, etc.
Your response should follow the following form: wrap your improved prompt using the <PROMPT></PROMPT> tag to wrap your point-by-point improved prompt and don't respond with anything else but the prompt,
eg: <PROMPT>new prompt</PROMPT>.Make sure your output is strictly semantically consistent with the query prompt mentioned by the user.point-by-point. just respond with the <PROMPT> tag, don't add any other content.
"""

prompt_engineer_re_improve = """
Now that you've got feedback from the user, who has made a suggestion about the response you just gave: {{suggestion}}, you'll want to read the user's suggestion carefully and modify your improved prompt to wrap your response using the <PROMPT></PROMPT> tag.
"""

prompt_engineer_prompt = """
Now, the user's query prompt is {{prompt}}, give me your response and wrap it in <PROMPT></PROMPT> tag.Respond in English.
"""

pretend_user_pre_system = """
You are a senior biomedical expert, and you analyze two descriptions to see if they match semantically. Please critically judge the narratives in your area of expertise and give the results of your judgment. Your response should follow the following format: use <RESULT></RESULT> tag to wrap the result of your judgment, and do not reply with anything else but the <RESULT> tag. You can only reply with <RESULT>YES</RESULT> for a match or <RESULT>NO</RESULT> for a no match.
"""

pretend_user_pre_prompt = """
Now, the two descriptions are {{prompt1}} and {{prompt2}}, give your judgment. use <RESULT>YES</RESULT> or <RESULT>NO</RESULT> without any other content.
"""

pretend_user_system = """
You are a senior biomedical professional who can identify semantic differences in the context of your specialty, give suggestions for adjustments, and respond point by point to two different descriptions. Your response should follow the following form: use <RESULT></RESULT> tag to wrap your difference analysis results and give suggestions for adjustments, note: semantics becoming rich and details becoming full is not a difference, only the meaning of the professional context changes to be included in the difference analysis. Don't respond with anything other than the <RESULT> tag, eg: <RESULT>1. Discrepancy 1 and recommendations; 2. Discrepancy 2 and recommendations</RESULT>.
"""

pretend_user_prompt = """
Now, given two descriptions: prompt1: {{prompt1}} and prompt2: {{prompt2}}, give me the results of the discrepancy analysis and wrap them in <RESULT></RESULT> tag, you need to treat prompt1 as the truth in order to judge the discrepancy of prompt2 and give a recommendation. You need to call the writer of prompt2 "you" and the writer of prompt1 "user".
"""

file_extract_analyst_system = """
You are an expert in file analysis with a biomedical background, and you can extract all the filenames mentioned in a user's request and stitch them together using the python list format. Your response should follow the following format: wrap your extracted list of filenames in <FILE></FILE> tag and don't reply with anything else but the <FILE> tag, eg: <FILE>["file1.txt", "file2.md"]</FILE>.
"""

file_extract_analyst_prompt = """
Now, the user request is {{prompt}}, give me the list of files you extracted and wrap them with <FILE></FILE> tag.
"""

data_further_analyst_system = """
You are a data analyst with a biomedical background and now, in response to a user's request for a specific file name, you want to analyze whether the file name and the suffix need further parsing. If the file is a non-text file or an unformatted file or a fixed format file in the biomedical domain, no further parsing is required. Otherwise, its internal format needs to be parsed. You only need to analyze if further parsing is needed. Your response should follow the following format: use the <RESULT></RESULT> tag to wrap your judgment, and do not reply with anything other than the <RESULT> tag. You can only use <RESULT>YES</RESULT> to indicate that further parsing is required, or <RESULT>NO</RESULT> to indicate that no further parsing is required.
Keep in mind that what you are trying to do is to make a judgment about the file format without having to consider the file semantics and specifics, and if a file's format is fixed, do not continue to analyze it.
"""

data_further_analyst_prompt = """
Now, the user request is: {{prompt}} and the file you need to analyze is {{file}}. Please answer <RESULT>YES</RESULT> or <RESULT>NO</RESULT>.
"""

data_analyst_system = """
You are a document analysis expert with a biomedical background, and for a given document, you can find the right way to analyze it to provide the most helpful information for the user's request. You have two tools to analyze the file, one is <TOOL>file_reader(m,n)</TOOL>, which reads the file from line m to line n and returns it to you for further analysis, if you use this tool you need to fill in m and n with specific numbers when answering, and the other is the intelligent analyzer <TOOL>file_agent</TOOL>, which can be used to help you when you are not quite sure of the file structure. clear about the file structure, you can use this tool to help you analyze it intelligently. Your responses should follow the following format: use <TOOL></TOOL> tag to wrap the tool of your choice, and don't reply with anything other than the <TOOL> tag. For text-like files, try to choose the file_reader tool.
Note that for non text formats such as zip, gz, png, etc., please do not use the file_reader tool.Please note that for non text formats such as zip, gz, png, etc., file agent should be directly used for analysis.
"""

data_analyst_prompt = """
Now, the user request is {{prompt}} and the name of the file you want to analyze is: {{file}}, give me your answer and wrap it in <TOOL></TOOL> tag.
"""

data_analyst_analyse = """
Now, you've got lines {{m}} through {{n}} of the file {{file}} with the following content:【{content}】. You have to determine whether the given content is sufficient and analyze the file format according to the user's request {{prompt}}, and make a rigorous and concise description of the file format related to the user with the obtained data. Your response should follow the following form: if your analysis results are sufficient to describe the file format perfectly, use <FORMAT></FORMAT> tag to wrap your condensed file format and description of useful information, and don't reply with anything else but <FORMAT> tag. If the given file content is not enough to determine the file format, use the <TOOL> tag, use file_agent or adjust the values of m,n in file_reader to re-invoke the tool.
"""

file_agent_system = """
You are an expert in intelligent analysis of documents with a biomedical background. For user requests and related documents, you can add useful information based on the description and suffix of the document, such as what needs to be done subsequently and how the document was processed, its format, and what to look for. Your responses should follow the following format: use <ANALYSE></ANALYSE> tag to wrap the results of your analysis, and do not respond with anything other than <ANALYSE> tag.
Please analyze the contents of the file based on the file description and suffix, only the file needs to be considered, not any other task.
"""

file_agent_prompt = """
Now, the user request is: {{prompt}} and the file you need to analyze is {{file}}, give me your response and wrap it with the <ANALYSE></ANALYSE> tag.
"""

tool_invocation_expert_system = """
You are a tool invocation expert with a biomedical background, and you can form a natural language description of the tool's features, usage, parameters, and so on, through the tool's documentation, outputting a descriptive paragraph, making sure that your description matches the tool's original documentation. Your response should follow this format: wrap your descriptive paragraph in <REDESCRIPTION></REDESCRIPTION> tag, and do not respond with anything other than <REDESCRIPTION> tag.
"""

tool_invocation_expert_prompt = """
Now, the name of the tool you want to describe is: {{toolname}} and its documentation is: {{documentation}}. Give me your response and wrap it with the <REDESCRIPTION></REDESCRIPTION> tag.
"""

tool_expert_system = """
You are a tool expert with a background of biomedical knowledge, and you have to design a necessary tool in response to a user request and give the tool's features, use, parameters, etc., described in a natural language paragraph. Your response should follow the following format: use <TOOL></TOOL> tags to wrap the tool you designed and do not reply with anything else but the <TOOL> tag. You only need to design one tool, and you don't need to make sure that the tool functionality can completely cover all the tasks requested by the user.
"""

tool_expert_prompt = """
Now, the user request is {{prompt}}, The file details mentioned by the user are: {{file_analyse_result}}. Here is a list of tool names for your reference: {{tool_list}}. You need to describe in natural language what the tool does and the expected results, give me your response and wrap it with the <TOOL></TOOL> tag.
"""

tool_expert_more = """
Now, design another tool, making sure that you don't duplicate the functionality of the tool you answered earlier. give me your response and wrap it with the <TOOL></TOOL> tag.
"""


tool_scorer_system = """
You are a professional tool scorer with biomedical background knowledge, and you can score each tool based on the user's request. The score represents the degree of relevance of the tool to the request, with a score range of 1-10, where 1 represents completely unrelated and 10 represents significant relevance. Along with the tool documentation, you'll also get a description of the tool's functionality provided by the tool's analysts as a reference. Your response should follow the following format: use <SCORE></SCORE> tags to wrap the score you are scoring , use <REASON></REASON> tags to wrap the reason for your scoring, do not use anything other than the <SCORE> and <REASON> tags for your response.
Notice: If the tool is judged to be useful as a sub-set of problem solving, then the tool is also useful and should be given a higher score.
# The score should be an integer。
"""

tool_scorer_prompt = """
Now, the user request is: {{prompt}} and the name of the tool you want to score is: {{toolname}} and its documentation is: {{documentation}}. Give me your response and wrap it with the <SCORE></SCORE> tag and the <REASON></REASON> tag.
"""

workflow_architect_system = """
You are a program architect. You need to combine the given tools into workflow to fulfill the user's request and design the workflow logic based on the user's request and the given list of tools. Your reply should follow the following format: wrap your workflow in <RESULT></RESULT> tags that describe the overall logic of the workflow. Do not reply with anything other than the <RESULT> tag.
# Notice: workflow should conform to the following format: be core oriented to tool usage, be sorted by problem solving sequence, and clearly define the data drivers as well as the invocation process of the tool.
# Notice: You don't need to use all the tools given, just make sure that the data link of the workflow you are designing is complete!
# As an architect, if there are processes that aren't supported in the tool, but you're confident that you can write the code to implement them yourself, you can state them in the workflow (with None in the corresponding tool and a task definition in the description).
# For simple requests, you can complete the task by writing Python code without using any tools.
# Note that for each stage of the workflow, the return value can only be a common python data type, not a third-party library type or python class (e.g., dataframe for pandas, numpy)
"""

workflow_architect_prompt = """
Now, the user request is: {{prompt}}, The extra information about the file mentioned by the user is {{analysis_result}}, and the list of tools you can to use is: {{tool_list}}. Give me your workflow response and wrap it with the <RESULT></RESULT> tag.
# Notice: Please note that the external tools you are using should be in the given tool list. Of course, For pre/post-processing of tools and articulation between tools, you can use a `GOAL` to describe them.
# Notice: Please note that your workflow design should conform to the order of calls, and carefully check the description of the tools to ensure that you are clear about the input and output of each tool.
# Notice: You need to intelligently integrate tools based on your strong analytical and comprehension abilities to form a problem-solving workflow.


this is a format example for request: I want to calculate (2+3)*5=? and the tools are Addition and Multiplication:
\"\"\"
<RESULT>The workflow can be constituted using the following tools in sequence:
1. **Goal**: calculate 2+3
    - **Tool**: Addition
    - **Input**: 2, 3
    - **Output**: 5
    - **Description**: Perform addition operation on the input numbers.
2. **Goal**: multiply the result by 5
    - **Tool**: Multiplication
    - **Input**: 5, 5
    - **Output**: 25
    - **Description**: Perform multiplication operation on the input numbers.
</RESULT>
\"\"\"
You cannot use tools outside of the list. As a professional architect, you need to utilize these tools to the fullest of your abilities
# Notice: In the tool field, you can only select one or more existing tool names.
# Notice: User data is often not something that a single tool can solve. You need to carefully and rigorously consider the input, output, and functional descriptions of each tool, and connect them reasonably.
# Each tool may not directly solve the problem completely, but the combination of tools often plays a role.
# Note that the Input and Output of each stage should be python types such as strings, numbers, lists, etc. or file paths, not Dataframe/numpy array etc.
# Note that sometimes data preprocessing is unnecessary. Encourage the use of file interactions between GOALs.
# Note that just design a workflow, you are an architect and don't need to give me any specific code
"""

file_appendix_prompt = """
【The {{type}} of the file {{name}}, you can refer to the design of workflow according to the content of the file and the format of the file,
the content of the file is as follows
===============
{content}
===============】
"""

bioinformatician_system = """
You are a bioinformatician and your task is to provide guidance on how the tool should be involved in the problem solving process for a request mentioned by a user and given a list of tools. You have to mention in the description of your response, [what the tool uses as an antecedent input, what things the tool can do, and what the tool can do to help with the subsequent output]. Your reply is going to follow the following format: use <DESCRIPTION></DESCRIPTION> tags to wrap your guideline suggestions and don't reply with anything else but <DESCRIPTION> tags, eg: <DESCRIPTION>{description}</DESCRIPTION>.
# Notice: Please provide a rigorous and objective evaluation of the tool's assistance to user requests. If the tool is not very helpful in solving the problem, please point it out in your response and state the reasons for not recommending it
"""

bioinformatician_prompt = """
Now, the content of the user's request is {{prompt}} and the tool is {{toolname}} and its documentation is: {{documentation}}. Give me your response and wrap it with the <DESCRIPTION></DESCRIPTION> tag.
"""

tool_rescorer_system = """
You are a bioinformatics expert who can use your extensive knowledge to analyze biomedical problems. You need to re rate a tool based on user requests and the original scoring and usage suggestions of a tool list. You need to carefully analyze the usage suggestions of the original tool list, reconsider the contribution of a given tool, and output a new score and explain the reasons. The score represents the degree of relevance of the tool to the request, with a score range of 1-10, where 1 represents completely unrelated and 10 represents significant relevance. Your reply should follow the following format: use the<SCORE></SCORE>tag to package your score, use the<REASON></REASON>tag to package your scoring reason, and do not use any other content to reply except for the<SCORE>and<REASON>tag.
# Note：Tools may not necessarily cover all the needs of users, as long as they meet the needs of a certain local link, they are also very useful.
# For example, the file you want to score may not be directly related to user input and requirements, but its existence brings convenience to intermediate file processing, so its relevance is also significant.
# The score should be an integer.
"""

tool_rescorer_prompt ="""
Now, the user request is: {{prompt}} and the original list of high scoring tools and their usage suggestions are as follows: {{origin_score}}. The name of the tool you want to score is: {{toolname}} and its documentation is: {{documentation}}. Give me your response and wrap it with the <SCORE></SCORE> tag and the <REASON></REASON> tag.
"""

format_desinger_system = """
You are a bioinformatics expert, and your job is to break down the text of a divided workflow into multiple stages and convert it into a formal and standardized format based on user requests. You need to use rigorous and careful thinking to ensure semantic invariance before and after the conversion. Your response should follow the following format: use the<STAGE></STAGE>tag for one stage, and use multiple consecutive<STAGE>tags to form the original workflow, without replying to anything other than the<STAGE>tag.

this is a format example for query: I want to calculate (2+3)*5=? and the workflow is:
\"\"\"
<RESULT>The workflow can be constituted using the following tools in sequence:
1. **Goal**: calculate 2+3
    - **Tool**: Addition
    - **Input**: 2, 3
    - **Output**: 5
    - **Description**: Perform addition operation on the input numbers.
2. **Goal**: multiply the result by 5
    - **Tool**: Multiplication
    - **Input**: 5, 5
    - **Output**: 25
    - **Description**: Perform multiplication operation on the input numbers.
</RESULT>
\"\"\"

your response should be like:
\"\"\"
<STAGE>**Goal**: calculate 2+3
    - **Tool**: Addition
    - **Input**: 2, 3
    - **Output**: 5
    - **Description**: Perform addition operation on the input numbers.
</STAGE>
<STAGE>**Goal**: multiply the result by 5
    - **Tool**: Multiplication
    - **Input**: 5, 5
    - **Output**: 25
    - **Description**: Perform multiplication operation on the input numbers.
</STAGE>
\"\"\"
"""

format_desinger_prompt = """
Now, the user's request is {{prompt}}, and the resulting workflow is {{workflow}}. You need to divide the workflow into multiple stages and respond in a fixed format. Give me your response and wrap it with the <STAGE></STAGE> tag.
"""

tool_extract_analyst_system = """
You are a tool expert, and you can determine which tools are used in the workflow based on the planned workflow and an existing workflow list, and you need to return the list of tools used in the workflow. Your response should follow the following format: wrap each tool with a<TOOL></TOOL>tag and do not reply to anything other than the<TOOL>tag.


this is a format example for query: I want to calculate (2+3)*5=? and the workflow is:
\"\"\"
<RESULT>The workflow can be constituted using the following tools in sequence:
1. **Goal**: calculate 2+3
    - **Tool**: Addition
    - **Input**: 2, 3
    - **Output**: 5
    - **Description**: Perform addition operation on the input numbers.
2. **Goal**: multiply the result by 5
    - **Tool**: Multiplication
    - **Input**: 5, 5
    - **Output**: 25
    - **Description**: Perform multiplication operation on the input numbers.
</RESULT>
\"\"\"

your response should be like:
\"\"\"
<TOOL>addition</TOOL>
<TOOL>multiplication</TOOL>
\"\"\"

# Notice: Please note: custom script is not included in the statistics.
# Notice: Please remember to ignore the custom script and do not include it in the response
"""

tool_extract_analyst_prompt = """
Now, the user's request is {{prompt}}, and the resulting workflow is {{workflow}}. The tool list is {{tool_list}}. You need to refer to the tool list and return the tools used by the workflow to me.Give me your response and wrap it with the <TOOL></TOOL> tag.
"""

tool_suggestion_system = """
You are a bioinformatics expert, and your task is to design a workflow for user requests and solutions. Think about how to apply the tools mentioned in the workflow, and you will receive a list of tools and functional descriptions for each tool. You need to think about how the tools should be specifically applied in the workflow stage, and provide suggestions and precautions for tool use. Your response should follow the following format: wrap your analysis results with the<SUGGESTION></SUGGESTION>tag, and do not reply to anything other than the<SUGGESTION>tag.
# Attention: When giving advice, you should carefully read the description of each tool and ensure that you provide answers after carefully understanding the workflow.
"""

tool_suggestion_prompt = """
Now, the user's request is {{prompt}}, and the complete process of the designed workflow is {{workflow}}. Based on the tool description {{tool_info}} below, provide suggestions for using the tool {{tool}}. Give me your response and wrap it with the <SUGGESTION></SUGGESTION> tag.
# Remember, you can only provide suggestions for the given tool {{tool}}, and everything should be answered specifically for the tool {{tool}}.
"""

action_architecture_expert_system = """
You are an architecture expert, and your task is to divide sub stages and expand descriptions based on user requests and designed workflow. Your answer is a detailed description of the details, planning, implementation route, and expected goals of a certain sub stage. It is important to delve into the details to ensure accurate alignment with the workflow situation. Your response should follow the following format: wrap your analysis results with the<STAGE></STAGE>tag, and do not reply to anything other than the<STAGE>tag.
"""

action_architecture_expert_prompt = """
Now, the user's request is {{prompt}}, the full workflow of the plan is {{workflow}}, and the tool's recommendation for use is {{tool_suggestion_data}}, and you want to give a detailed description of the details, the planning, the route of fulfillment, and the desired goal for the sub-phase {{stage}}.
"""

programmer_system = """
You are a programmer with a background in bioinformatics. Your task is to write Python code based on the tool's description, user requests, designed workflow, and specific subtasks you are responsible for. You need to understand the entire workflow process, but when implementing modern code, you only need to implement the subtasks you are responsible for to ensure rigor and meticulousness, so that the code can complete the tasks. Your response should follow the following format: use<CODE></CODE>tag to package your code, and do not reply to anything other than the<CODE>tag.
# Note: The code implementation should be written as a python function with the function name {{name}} and the inputs and outputs should be designed according to the requirement.
# Note: All tools are a python function and are imported into the environment variable by default, you can call them directly by name as well as by input.
# No need to write large comments, focus on code implementation.
# For example, for a table operation, instead of returning the dataframe in the function, you should save the dataframe as a file and return the file path.
"""


programmer_prompt = """
Now, the user's request is {{prompt}}, the designed workflow is {{workflow}}, the list of tools and suggestions for their use are as follows: {{tool_suggestion_data}}, and the subtasks you're responsible for are {{subtask}}.The name of the function you want to implement is {{name}}.

# Notice: Please note that your core task is to use the tool, write the code and solve the problem, the code you return must be the full implementation without any placeholders.
# Notice: You have to write the tool call process as code and respond.
# Notice: Note that your results have to be returned explicitly using the return statement, and print is unnecessary.
# Notice: For the given tool, you do not need to import packages.
# Notice: The tools are provided in the form of Python functions and can be directly called.
# You only need to write the function body, not the function call code!
# Remember that you want to implement the functionality in the cleanest way possible, without adding other unnecessary processing and checks!
It is forbidden to return data types such as lists, which you have to write the subsequent processing code to store the data as a file!!!
For example, pandas and numpy need to be converted to a list type.
# Your return format must be saved as a file, which can be a two item tuple of the file and its explanation
Give me python function and wrap it with the <CODE></CODE> tag.
"""

programmer_func_fix = """
Your code did not pass the code review because you did not provide a specific implementation for a function: {{function}}. Please correct your Python code, carefully complete the function, wrap the code in the<CODE>tag, and provide me with your response.
# If it's a third-party library for Python, don't forget to import it.
# Remember: you can only write the function body, not the function call code!
# Your function must be named as specified, and name changes are prohibited!
"""

programmer_fail_test = """
Your code didn't pass the test, the test engineer gave you test suggestions, you came to rewrite the code based on the test engineer's suggestions, taking care to learn from the suggestions given by the test. The test suggestion is {{suggestion}}.
Give me python function and wrap it with the <CODE></CODE> tag.
# Remember: you can only write the function body, not the function call code!
# Your function must be named as specified, and name changes are prohibited!
"""

code_reviewer_system = """
You are a code reviewer, your task is to judge the quality of the programmer's code and whether it accomplishes the expected goals based on the user's request, the design workflow, the tool list description, the subtasks the programmer is responsible for, and the implemented code, you have to carefully check whether the programmer has unfinished areas or places where placeholders are used instead, and if there are any items that do not match with the function you should also raise it. Your response should follow this format: use <RESULT></RESULT> tags to wrap the results of your judgment (<RESULT>YES</RESULT> for programmers who passed the code review or <RESULT>NO</RESULT> for programmers who did not pass the code review) and <REASON></REASON> tags to wrap the reason for the judgment, and <REASON></REASON> tags to wrap the reason for the judgment. REASON> wraps the reason for the judgment. Do not reply with anything other than the <RESULT> tag and the <REASON> tag.
# Notice: Note that the tool definitions are all implemented and imported into the development environment by default, you just need to determine the programmer's code logic and the correctness of using the tool.The code does not need to add additional import statements.
# The programmer is responsible for some of the subtasks, so the code may not use all of the tools. You just need to check that the code does not accomplish the subtasks as expected and that there are no occurrences of functions that are not defined in the tool list or mismatched inputs and outputs.
# What you're evaluating is a function, and functions don't have to be called specifically, they just have to have implementation logic.
"""

code_reviewer_prompt = """
Now, the user's request is {{prompt}}, the designed workflow is {{workflow}}, the description of the tool list is {{tool_suggestion_data}}, the programmer's subtask is {{subtask}}, the programmer's implementation of the tool function is {{code}}, and you're going to do a code review of the programmer's code. Perform a code review.Give me your response and wrap it with the <RESULT></RESULT> tag and the <REASON></REASON> tag.
# Notice: The given list of tools doesn't need to be imported, don't think about the details of the IMPORT in the REVIEW.
Note that regarding the fact that the tool function {{tool_list}} is already implemented externally and can be used without importing, you should not mention this part in your code review.
"""


code_reviewer1_system = """
You are a code reviewer, and your task is to check the programmer's code. Programmers need to use a given list of tools reasonably according to requirements to implement tasks, but occasionally they may use functions that are not mentioned in the list, nor defined or imported. In this case, you need to check these functions. Of course, for a given list of existing tools, you don't need to check. Your response should follow the following format: wrap your judgment results in<RESULT></RESULT>tags (<RESULT>YES</RESULT>indicates that the programmer has passed code review or<RESULT>NO</RESULT>indicates that code review has not been passed), as well as several <TOOL></TOOL>tags, each <TOOL> tag containing an unimplemented function name.
# Note: Your core work is to check the functions called in the code, without checking whether the list of tools I give you is implemented.
"""

code_reviewer1_prompt = """
Now, the programmer's implementation of the function is {{code}}.Implemented tools are {{tool_list}}.
Give me your response and wrap it with the <RESULT></RESULT> tag and the <TOOL></TOOL> tags.
"""

tester_system = """
You are a testing engineer, and your goal is to write a unit test function corresponding to the subtasks in the workflow and the code implemented by the programmer. Your test function should be able to verify the correctness of the code implemented by the programmer and ensure that the test code covers the functionality of the functional code. Your response should follow the following format: use<CODE></CODE>tag to package your analysis results, and do not reply to anything other than the<CODE>tag.
"""

tester_prompt = """
Now, the user's request is {{prompt}}, the designed workflow is {{workflow}}, the description of the tool list is {{tool_suggestion_data}}, the programmer's subtask is {{subtask}}, the programmer's implementation of the function is {{code}}.
Your test function name is: {{function}}.

# Note that the tools in the tool list do not need to be imported separately, as they have already been imported into the environment by default.
\"\"\"
Your test function should be like:
<CODE>
def test0():
    # prepare the input data
    try:
        output = action0(input)
    except Exception as e:
        return False, repr(e)
    return True, output
</CODE>
\"\"\"
Your test function should return two values, the first is whether the test passes, a boolean value; if it passes, the second value is the return value from the execution of the programmer's code you called; if it doesn't pass, the second value is why it didn't pass.
# Don't use mechanisms such as mock to simulate the implementation of the tool, trust that the tool is already implemented.
# Don't forget to import common python third-party libraries.
# Prohibit the use of mock.
Note that the resource pool you use to write your test function are as follows: {{resource_pool}}, you can't make up your own files to test them, you have to choose the right ones to use in the given resources.
Give me python function and wrap it with the <CODE></CODE> tag.
# Please carefully understand the content of the resource pool and make reasonable use of it
# Note that you only need to provide a function definition and do not need to write calling code!
# It is forbidden to create your own files when writing test code, and all the file resources you use must come from the resource pool!
# You are prohibited from creating test data. You can only call the action code with resources from the resource pool
# Your code must not do anything extra other than call the action, and must be kept very simple.
# You can read the resource pool file in the code and prohibit the use of any [example] and [mock] behaviors!
# Attention, it is forbidden to fabricate data and try to use data from the resource pool. You can write code to read files
"""


tester_rethink = """
Your code was executed, but the result returned False, you want to examine the action code and infer from the error message why the error was reported and give your suggestions for improvement. Your response should follow the following format: use <REASON></REASON> to wrap the reason for the judgement and suggestions for improvement. Do not reply with anything other than the <REASON> tag.
The error message is: {{info}}.
"""

tester_error = """
The code reports an error, and you check to see if there is a problem with your test code (e.g., the formatting isn't aligned or there is a missing library import) or if there is a problem with the programmer's original program code. Your response should follow this format: use <RESULT></RESULT> tags to wrap the results of your judgment (<RESULT>YES</RESULT> to indicate that there is a problem with your test code or <RESULT>NO</RESULT> to indicate that there is a problem with the programmer's original code), and <REASON></REASON> to wrap the judgment Reason. Do not reply with anything other than the <RESULT> tag and the <REASON> tag.
The error message is: {{info}}.
"""

tester_fix = """
Below, reimplement your test code based on the above summary, taking care to avoid the problems mentioned above.
Give me the test code and wrap it with the <CODE></CODE> tag.
# Remember: you can only write the function body, not the function call code!
# Your function must be named as specified, and name changes are prohibited!
"""

tester_fail_test = """
There are some problems with your code implementation, here are the suggestions for changes, you come to rewrite the code according to the suggestions, be careful to absorb the content of the suggestions. The suggestion is {{suggestion}}.
Give me the test code and wrap it with the <CODE></CODE> tag.
# Remember: you can only write the function body, not the function call code!
# Your function must be named as specified, and name changes are prohibited!
"""

tester_description = """
The test passed, and the test program returned the following result:{{output}}, Now, you write a description of the program written by the programmer, describing what the programmer's code does, and a description of the contents of the return value, with a bit of labeling for the type of the return value. Your response should follow this format: wrap the description of the programmer's code in <DESCRIPTION></DESCRIPTION> tags, wrap the description of the content of the return value in <OUTPUT></OUTPUT> tags, and annotate the type of the return value with <TYPE></TYPE> tags (note that if the return value is a filename or path information, the type is `file`), and do not reply with anything other than <DESCRIPTION> and <OUTPUT> and <TYPE> tag.
# You can only use a set of <DESCRIPTION> and <OUTPUT>, <TYPE> tag.For example, if a code returns the following data (1, “out.txt”, “exp.png”), you can mark the <TYPE> tag with [dict] instead of multiple type tags.
"""

file_description_system = """
You are an expert in file analysis, and you can provide a one-paragraph descriptive description of the file that provides only useful information and does not need to contain your guesses or subjective comments, and your response should follow the following format: use <FILE></FILE> tag to wrap the results of your analysis, and do not reply with anything else but the <FILE> tag.
"""


file_description_prompt = """
Now, you want to make a paragraph description of the file {{file}} based on the user's request: {{prompt}}.
Give me your response and wrap it with the <FILE></FILE> tag.
"""

summary_system = """
You are a biomedical architect, you are a member of a group of agencies, your task is to address the user's needs, agencies to design the process, the implementation of the process of the resources generated by the user to give a summary report and response, your report should be a comprehensive answer to the user's questions, to be very professional, reflecting the quality of the biomedical experts, you can use the markdown in the right place! Insert pictures or tables. Your response should follow the following format: use <SUMMARY></SUMMARY> tag to wrap your summary report, and do not respond with anything other than <SUMMARY> tag.
"""

summary_prompt = """
Now, the user's request is {{prompt}}, agents' process planning is {{workflow}}, and the resource pool obtained from executing the process is {{resource_pool}}. You give the summary report.
Give me your response and wrap it with the <SUMMARY></SUMMARY> tag.
# Notice: Note that your report will only be shown to the user alone, so there is no need to think about confidentiality, it has to be comprehensive and also logical in terms of communicating with the user.
# Note: To highlight the conclusion data, the core structure of your report content is to answer the user's question.
# Please make sure you are objective and truthful, and make the most of the descriptions in the resource pool.
"""


mermaid_system = """ You are an experienced process mapper who can perfectly translate natural language descriptions of task processes into code that conforms to the mermaid syntax. Your task is: for a given workflow, you rigorously conform its semantics into mermaid code.
# Note that it is not necessary to use the {```mermaid} beginning as well as the {```} ending in the mermaid code, just answer the body of the code directly.
# You can only use arrow syntax, here is an example where you should replace ABCD with a semantic string:
<CODE>
graph LR
    A --> B
    B --> C
    B --> D
</CODE>
"""

mermaid_prompt = """
Currently, the workflow is described as {{workflow}}, which you translate into mermaid code.
Give me your response and wrap it with the <CODE></CODE> tag.
# Please note that only the most basic syntax needs to be used, not advanced features such as class statements, Dataframes, etc.
"""


necessary_system = """
You are a bioinformatician, and your job is to determine, for a sub-task of the overall workflow, whether the sub-task is an unskippable task, such as for machine learning, where front-loaded data processing is not a necessary task. Your response should follow the following form: wrap your judgment using <RESULT></RESULT> tags, (<RESULT>YES</RESULT> for the subtask is unskippable or <RESULT>NO</RESULT> for the subtask is skippable) and do not respond with anything else but <RESULT> tags.
"""

necessary_prompt = """
Now, the workflow designed for the user's request {{prompt}} is: {{workflow}}, and you need to determine whether {{subtask}} is unskippable.
Give me your response and wrap it with the <RESULT></RESULT> tag.
"""

split_again_system = """
You are a bioinformatician and your job is to determine, for a subtask of the overall workflow, whether the subtask can be split again into two more specific tasks. Your response should follow the following format: wrap your judgment in <RESULT></RESULT> tags, (<RESULT>YES</RESULT> for the subtask can be split again or <RESULT>NO</RESULT> for the subtask can't be split) and don't respond to anything else but the <RESULT> tags.
"""

split_again_prompt = """
Now, the workflow designed for the user's request {{prompt}} is: {{workflow}}, and you need to determine whether {{subtask}} is detachable.
Give me your response and wrap it with the <RESULT></RESULT> tag.
"""

workflow_fix_system = """
You are a program architect and you can combine the given tools into a workflow to satisfy the user's request based on the user's request and the given list of tools and design the workflow logic. Now, you have to give a suggestion for fixing the workflow based on the workflow that has been designed and the errors encountered while executing the workflow. Your response should follow the following format: wrap your fix suggestion in <SUGGESTION></SUGGESTION> tag and don't reply with anything other than <SUGGESTION> tag.
"""

workflow_fix_ignore = """
Now, the user's request is :{{prompt}}, the designed workflow is {{workflow}}, and I encountered an error report {{error}} while executing subtask: {{subtask}}, now, I want to skip this subtask, and you need to give fix suggestions.
Give me your response and wrap it with the <SUGGESTION></SUGGESTION> tag.
"""

workflow_fix_split = """
Now, the user's request is :{{prompt}}, the designed workflow is {{workflow}}, and I encountered an error report {{error}} while executing subtask:{{subtask}}, now, I want to split this subtask into two clearer and simpler tasks, and you need to give fix suggestions.
Give me your response and wrap it with the <SUGGESTION></SUGGESTION> tag.
"""

workflow_fix_normal = """
Now, the user's request is :{{prompt}}, the designed workflow is {{workflow}}, while executing subtask:{{subtask}} I encountered an error {{error}}, now, I want to make some adjustments to the workflow in response to the reported error, you need to give fix suggestions.
Your tweak is going to be based on the original workflow, but make a drastic simplification of it by removing as much as possible all file preprocessing and post-processing items.
Give me your response and wrap it with the <SUGGESTION></SUGGESTION> tag.
"""

re_workflow_system = """
You are a program architect and you can combine the given tools into workflow to fulfill the user's request and design the workflow logic based on the user's request and the given list of tools. Now, you want to regenerate a new workflow that meets the recommendations based on the workflow you've already designed and the suggested fixes for that workflow.Your response should follow the following format: wrap your modified workflow in <RESULT></RESULT> tags, and don't respond with anything other than the <RESULT> tags. content.

this is a format example for query: I want to calculate (2+3)*5=? and the workflow is:
\"\"\"
<RESULT>The workflow can be constituted using the following tools in sequence:
1. **Goal**: calculate 2+3
    - **Tool**: Addition
    - **Input**: 2, 3
    - **Output**: 5
    - **Description**: Perform addition operation on the input numbers.
2. **Goal**: multiply the result by 5
    - **Tool**: Multiplication
    - **Input**: 5, 5
    - **Output**: 25
    - **Description**: Perform multiplication operation on the input numbers.
</RESULT>
\"\"\"
"""

re_workflow_prompt = """
Now, the user request is: {{prompt}}, The extra information about the file mentioned by the user is {{analysis_result}}, and the list of tools you can to use is: {{tool_list}}.
The workflow that has been designed is {{workflow}} and you have to give a modified workflow based on the given suggestion {{suggestion}}.
Prohibit the use of extra fabricated tools when redesigning workflows!
Give me your workflow and wrap it with the <REASON></REASON> tag.
"""

request_analyse_system = """
You are a computer specialist with a background in bioinformatics, and your task is to write an appropriate request prompt for the user's overall behavior based on information about the tool the user is using, the code that was written to use the tool, and the inputs and outputs of the code.Your response is to follow the following format: use a <PROMPT></PROMPT> tag to wrap the results of your analysis and do not reply to anything else except the < PROMPT> tag do not reply with anything else.
# Note that the user's request language is Chinese.
"""

request_analyse_prompt = """
Now, the name of the tool used by the user is {{tool_name}}, the content of the tool documentation is {{tool_doc}}, the code to use the tool is {{tool_code}}, and the input and output information is as follows:{{data_info}}, and you need to write an appropriate request prompt for the user's overall behavior.

Below are some examples of user's requests, you can refer to its format but not to the content:
For tool vcf_to_maf：<PROMPT>我有一个突变的vcf文件{GJX-lung.pass.vcf}，利用vcf2maf进行变异注释生成maf文件</PROMPT>
For tool KOBAS_enrichment：<PROMPT>我有一个基因列表{PI3K-Akt.csv},利用KOBAS_enrichment进行基因集功能富集</PROMPT>
For tool MCPCounter：<PROMPT>我有一个表达谱文件{exp_ORAD.tsv}，利用MCPCounter进行免疫细胞定量分析</PROMPT>

Give me your response and wrap it with the <PROMPT></PROMPT> tag.
"""

workflow_analyse_system = """
You are a program architect with a biomedical background, and your task is to design a workflow logic based on user requests, tool documentation, tool calls, and test code, no need to provide implementation process, testing, and examples in the workflow. Your response is to follow the following format: use a <RESULT></RESULT> tag to wrap the results of your analysis and do not reply to anything else except the <RESULT> tag do not reply with anything else.

this is a format example for request: I want to calculate (2+3)? and the tool is Addition:
\"\"\"
<RESULT>The workflow can be constituted using the following tools in sequence:
1. **Goal**: calculate 2+3
    - **Tool**: Addition
    - **Input**: 2, 3
    - **Output**: 5
    - **Description**: Perform addition operation on the input numbers.
</RESULT>
# Note: There is no need for any content outside of workflow such as sample code, code implementation, and test in the <RESULT>
"""

workflow_analyse_prompt = """
Now, the user's request is {{question}}, the tool used are{{tool_name}}, the content of the tool documentation is {{tool_doc}}, and the code for calling and testing the tool is {{action_test_code}}. Please design the workflow logic based on this information.
"""

memory_tool_system = """
You are an expert in tool use with extensive biomedical knowledge. You have now used your existing library of tools to design a workflow to solve a problem based on a user's request for a problem and have successfully solved the problem. You are able to give the most accurate advice on the use of the tool based on the available information (including the user's request problem, the tool information, and the designed workflow), and as a summary of your experience, your experience should be concise and meaningful. Your response should follow the following format: wrap your analysis in a <SUGGESTION></SUGGESTION> tag and do not respond with anything other than <SUGGESTION> tag.
"""

memory_tool_prompt = """
Now, the user's original question is {{question}}, the planned workflow is {{workflow}}, and you now want to give your experience against the tool {{tool_name}}, and the tool documentation is {{tool_doc}}.
Give me your experience and wrap it with the <SUGGESTION></SUGGESTION> tag.
"""

memory_workflow_system = """
You are an expert in the use of tools with extensive biomedical knowledge. Now, you have used your existing library of tools to design a workflow to solve a problem based on a user's request problem and have successfully solved the problem. You are able to give the most accurate workflow planning experience based on the available information (including the user's request problem, and the designed workflow), and as a summary of your experience, your experience should be concise and meaningful. Your response should follow the following format: wrap your analysis in <SUGGESTION></SUGGESTION> tags and do not respond with anything other than <SUGGESTION> tags.
"""

memory_workflow_prompt = """
Now that the user's original question is {{question}}, and the planned workflow is {{workflow}}, you need to summarise the question's experience of planning against the workflow, making sure to keep it short and effective.
Give me your experience and wrap it with the <SUGGESTION></SUGGESTION> tag.
"""

translate_expert_system = """
You are a translation expert, and you can accurately translate the Chinese sentences I gave you into English sentences, ensuring that the sentence semantics remain unchanged rigorously.
"""

translate_expert_prompt = """
Now, the sentence I'm giving you is {{word}}.
Please provide me with your translation results and wrap it with the <WORD></WORD> tag.
"""

# Cell number 9

# scripts -> executor.py

import json
import redis
import time

#from config import Config
#from utils import generate_task_id, gprint, rprint

conn = redis.StrictRedis(decode_responses=True)

def push_code(code:str, test_func_name:str, config:Config, ensure_active=True, official=False):
    task_id = generate_task_id()
    conn.lpush(config.REDIS_EXECUTOR_LIST_TASK_KEY, json.dumps({
        "task_id":task_id,
        "task_path":config.task_path,
        "code":code,
        "test_func_name":test_func_name,
        "ensure_active":ensure_active,
        "official":official
    }, ensure_ascii=False))
    return task_id

def get_code_output(task_id:str, config:Config):
    while True:
        result = conn.hget(config.REDIS_EXECUTOR_LIST_DATA_KEY,task_id)
        if result is not None:
            return json.loads(result)
        time.sleep(config.EXECUTOR_CODE_WAITING_TIME)

# Cell number 10

# lab -> file_reader.py

def file_reader(name:str, path:str):

    MAX_CONTENT_SIZE = 2**12
    MAX_LINES = 5
    print(name,path)
    appendix = name.split(".")[-1]
    if appendix not in ["txt","csv","tsv"]:
        return False, None
    if appendix in ["csv","tsv"]:
        with open(path) as f:
            content = f.read()
            if len(content) < MAX_CONTENT_SIZE:
                return True, {
                    "type":"Full Content",
                    "content":content
                }

            content = ""
            f.seek(0)
            line_count = 0
            for line in f.readlines():
                if len(line) > MAX_CONTENT_SIZE*2:
                    return False, None
                content += line
                line_count += 1
                line_count
                if len(content) > MAX_CONTENT_SIZE or line_count >= MAX_LINES:
                    return True, {
                        "type":f"First {line_count} rows",
                        "content":content
                    }
        return False, None
    elif appendix == "txt":
        with open(path) as f:
            content = f.read()
            if len(content) < MAX_CONTENT_SIZE:
                return True, {
                    "type":"Full Content",
                    "content":content
                }
            else:
                return True, {
                    "type":"Partial summary",
                    "content":content[:MAX_CONTENT_SIZE]
                }

# Cell number 11

# scripts -> component.py

import os
import json
import time
import redis
import shutil
from typing import Any


#from config import Config
#from utils import yprint,generate_task_id

#from scripts.prompt import file_appendix_prompt
#from lab.file_reader import file_reader


class Status:
    def __init__(self, task_id:str, config:Config, update) -> None:
        self.data = {}
        self.task_id = task_id
        self.config = config
        self.update = update

    def __setattr__(self, name: str, value: Any) -> None:
        super().__setattr__(name, value)
        if name not in ["data","taskid","config","update"]:
            self.data[name] = value

    def __getattr__(self, name: str):
        try:
            self.data[name]
        except:
            return self.__getattr__(name)

    def get(self, name):
        return self.data.get(name)

    def show(self):
        yprint("="*30)
        print(self.data)
        yprint("="*30)

    def save(self, output_path='output.json'):
        with open(output_path,"w",encoding="utf8") as f:
            json.dump(self.data,f)

    def status_update(self, stage:str):
        if not self.update: return
        now = int(time.time())
        self.upload_status = {
            "stage":stage,
            "time":now
        }
        self.config.get_redis_connect().lpush(
            f"{self.config.REDIS_STATUS_LIST_KEY}:{self.task_id}",json.dumps({
                "uid":generate_task_id(),
                "stage":stage
            })
        )
        self.config.get_redis_connect().hset(
            self.config.REDIS_STATUS_DATA_KEY, self.task_id, json.dumps(self.data)
        )

class Task:
    def __init__(self, info:dict, config:Config, update=False, official=False) -> None:
        """
        info example:
        {
            "question":"this is a question",
            "files":[
                {
                    "name":"demo1",
                    "path":"/mnt/data/example/demo1"
                },
                {
                    "name":"demo2",
                    "path":"/mnt/data/example/demo2"
                }
            ]
        }
        """
        self.official = official


        question = info.get("question")
        files = info.get("files")

        self.question = question

        self.files = files
        self.config = config

        self.status = Status(self.config.get_task(), config, update)
        self.tool_manager = ToolManager(config)

        self.prepare()

    def prepare(self):
        self.raw_question = self.question
        task_id = self.config.get_task()
        task_path = os.path.join(
            f"{self.config.TASK_DIR}",self.config.time_path,task_id
        )

        self.config.task_path = task_path
        self.status.task_path = task_path
        self.status.raw_question = self.raw_question
        self.status.backtrack = []

        self.status.files = self.file_list

        self.file_appendix = ""

        if self.config.USE_FILE_APPENDIX:
            for f in self.files:
                ok, info = file_reader(f["name"],f["path"])
                if not ok:continue
                self.file_appendix += file_appendix_prompt.replace(
                    "{type}",info["type"]
                ).replace(
                    "{name}",f["name"]
                ).replace(
                    "{content}", info["content"]
                )
            with open("data/use_file_appendix.txt","w") as f:
                f.write(self.file_appendix)


        if not os.path.exists(task_path):
            os.makedirs(task_path)

        for file in self.files:
            print(file['path'])
            if not os.path.exists(file['path']):
                raise FileNotFoundError(f"no file:{file['path']=}")
            # os.system(f"cp {file['path']} {task_path}")
            shutil.copy(file['path'], task_path)
        #?###########################
        self.status.tools = {}
        tools = os.listdir(self.config.TOOL_DOC_DIR)
        # self.status.tools = {}
        for tool in tools:
            if tool.startswith("."):
                continue
            with open(os.path.join(self.config.TOOL_DOC_DIR,tool),"r",encoding="utf8") as f:
                document = f.read()
            self.status.tools[tool] = {
                "document":document
            }
        self.status.rescored = False
        self.status.code = {}

    def backtrack_update(self):
        self.status.backtrack.append({
            "workflow" : self.status.workflow,
            "tool_used": self.status.tool_used,
            "workflow_stages": self.status.workflow_stages,
            "resource_pool":self.status.resource_pool,
            "code_result":self.status.code_result,
            "code":self.status.code
        })
        self.status.tool_used = {}
        self.status.workflow_stages = []
        self.status.resource_pool = []
        self.status.code_result = {}
        self.status.code = {}

    @property
    def file_list(self):
        result = []
        for file in self.files:
            result.append(file['name'])
        return result

    @classmethod
    def fake_task(cls, config):
        fake_info = {"question":"","files":[]}
        return Task(
            fake_info, config
        )


class ToolManager:
    def __init__(self, config:Config) -> None:
        self.config = config
        self.r = config.get_redis_connect()

    def get_tool_list(self, ensure_active = True):
        if ensure_active:
            tools = self.r.hkeys(self.config.REDIS_ACTIVE_TOOL_KEY)
            return tools
        else:
            tool_ids = self.r.hkeys(self.config.REDIS_TOOL_INFO_KEY)
            return tool_ids


    def get_tool_info(self, tool_name):
        tool_id = self.r.hget(self.config.REDIS_ACTIVE_TOOL_KEY, tool_name)
        if not tool_id:
            return False, None
        tool_info = self.r.hget(self.config.REDIS_TOOL_INFO_KEY, tool_id)
        if not tool_info:
            return False, None
        return True, json.loads(tool_info)

    def get_tool_info_by_id(self, tool_id):
        tool_info = self.r.hget(self.config.REDIS_TOOL_INFO_KEY, tool_id)
        if not tool_info:
            return False, None
        return True, json.loads(tool_info)

    def get_tool_code(self, tool_name):
        ok, info = self.get_tool_info(tool_name)
        return info["tool_code"]

    def get_tool_doc(self, tool_name):
        ok, info = self.get_tool_info(tool_name)
        return info["tool_doc"]

    def get_tool_code_by_id(self, tool_id):
        ok, info = self.get_tool_info_by_id(tool_id)
        return info["tool_code"]

    def get_tool_doc_by_id(self, tool_id):
        ok, info = self.get_tool_info_by_id(tool_id)
        return info["tool_doc"]

    def get_tool_name_by_id(self, tool_id):
        ok, info = self.get_tool_info_by_id(tool_id)
        return info["tool_name"]

if __name__ == "__main__":
    config = Config()

    tool_manager = ToolManager(config)
    tools = tool_manager.get_tool_list()

    print(tools)
    print(type(tools))


# Cell number 12

# lab -> memory_retriver.py

import json
import time
#from config import Config
#from utils import generate_task_id

class TestMemoryRetriever:
    def __init__(self) -> None:
        self.r = Config().get_redis_connect()
        self.config = Config()

    def match(self, item_key, query_key, total_task_id, k=2):
        k = 3
        task_id = generate_task_id()
        data = {
            "task_id":task_id,
            "item_key":item_key,
            "query_key":query_key,
            "k":k,
            "threshold":0.4
        }
        self.r.lpush(self.config.REDIS_MEMORY_TASK_KEY,json.dumps(data))
        while True:
            result = self.r.hget(self.config.REDIS_MEMORY_RESULT_KEY, task_id)
            if result is None:
                time.sleep(1)
            else:
                result = json.loads(result)
                if len(result) == 0:
                    self.config.log_error("special", "result is None")
                    return ""
                content = []
                for item in result:
                    content.append(item["value"])
                    self.r.lpush(self.config.REDIS_MEMORY_LOG2, json.dumps({
                        "item_key":item_key,
                        "current_query_key":query_key,
                        "origin_query_key":item["key"],
                        "query_value":item["value"],
                        "memory_id":item["id"],
                        "origin_question_id":item["question_id"],
                        "current_task_id":total_task_id,
                        "k":len(result),
                        "score":item["score"]
                    }))
                content = f"\n{'='*10}\n".join(content)
                response = f"""
Here's a summary of some of your experiences and memories that you can use for reference: 【\n{content}\n】You must learn from experience and memory as well as reference!
"""
                return response



# Cell number 13

# agent.py

import asyncio
import json
import re

#from scripts.chat import ChatSession
#from scripts.component import Task
#from config import Config
#from scripts.executor import push_code, get_code_output
#from scripts.prompt import *
#from utils import rprint, gprint, get_action_code

#from lab.memory_retriever import TestMemoryRetriever


memory_retriever = TestMemoryRetriever()

class BaseAgent:
    system = "system prompt"

    actions_template = {
        "initial":{
            "prompt":"initial prompt is {{prompt}}",
            "keywords":["prompt",]
        },
        "someone":{
            "prompt":"{{value}} is good",
            "keywords":["value",]
        }
    }

    def __init__(self, task:Task) -> None:
        self.task = task
        self.config = task.config
        self.model = self.config.BASE_LLM_MODEL
        self.temperature = 0.2

    def format_template(self, action="initial", data=None, templates=None):
        if data is None:
            data = {}
        """
        data:
        {
            "prompt":"a prompt",
            "value":"some value"
        }
        """
        if templates is None:
            templates = self.actions_template
        template: str = templates.get(action)["prompt"]
        for item in data:
            template = template.replace("{"+item+"}", data[item])
        return template

    def create_chatsession(self, system=None):
        if system is None: system = self.system
        session = ChatSession(self.model, self.temperature, self.config)
        session.set_system(system)
        return session

    @staticmethod
    def status_update(key):
        def updater_decorator(func):
            def updater(self:BaseAgent, *args, **kw):
                response = func(self, *args, **kw)
                self.task.status.__setattr__(key, response)
                return response
            return updater
        return updater_decorator

    @staticmethod
    def retry(check_function=(lambda _:True),max_retry_times = 3, multi=False):
        def retry_decorator(func):
            def inner(*args, **kw):
                response = func(*args, **kw)
                retry_times = 1
                while retry_times < max_retry_times:
                    retry_times += 1
                    if not check_function(response):
                        response = func(*args, **kw)
                    else:
                        break
                else:
                    raise ValueError
                return response
            async def async_inner(*args, **kw):
                response = await func(*args, **kw)
                retry_times = 1
                while retry_times < max_retry_times:
                    retry_times += 1
                    if not check_function(response):
                        response = await func(*args, **kw)
                    else:
                        break
                else:
                    raise ValueError
                return response
            if multi:
                return async_inner
            return inner
        return retry_decorator


class ResponseHandler:
    @staticmethod
    def get_xml_tag_content(text,tag_name) -> str:
        results = re.findall(f'<{tag_name}>(.*?)</{tag_name}>', text, re.S)
        return results[0]

    @staticmethod
    def get_xml_tag_list_content(text,tag_name) -> str:
        results = re.findall(f'<{tag_name}>(.*?)</{tag_name}>', text, re.S)
        return results

    @staticmethod
    def assert_xml_tag_equal_handler(tag_name, equal_text, upper=True):
        def handler_decorator(func):
            def handler(*args, **kw):
                response = func(*args, **kw)
                content = ResponseHandler.get_xml_tag_content(response, tag_name)
                if upper: content = content.upper()
                return content == equal_text
            return handler
        return handler_decorator

    @staticmethod
    def chain_handler(chain_design_function):
        def chain_decorator(func):
            def handler(self:BaseAgent, *args, **kw):
                func(self, *args, **kw)
                return chain_design_function(self.task)
            return handler
        return chain_decorator

    @staticmethod
    def direct_chain_design(next_agent:BaseAgent, method:str):
        def chat_design_function(task):
            agent = next_agent(task)
            return agent.__getattribute__(method)
        return chat_design_function


    @staticmethod
    def xml_tag_content_handler(tag_name,multi=False):
        def handler_decorator(func):
            def handler(*args, **kw):
                response = func(*args, **kw)
                content = ResponseHandler.get_xml_tag_content(response, tag_name)
                return content
            async def async_handler(*args, **kw):
                response = await func(*args, **kw)
                content = ResponseHandler.get_xml_tag_content(response, tag_name)
                return content
            if multi:
                return async_handler
            return handler
        return handler_decorator

    @staticmethod
    def xml_tag_list_content_handler(tag_name,multi=False):
        def handler_decorator(func):
            def handler(*args, **kw):
                response = func(*args, **kw)
                content = ResponseHandler.get_xml_tag_list_content(response, tag_name)
                return content
            async def async_handler(*args, **kw):
                response = await func(*args, **kw)
                content = ResponseHandler.get_xml_tag_list_content(response, tag_name)
                return content
            if multi:
                return async_handler
            return handler
        return handler_decorator

    @staticmethod
    def multi_xml_tag_content_handler(multi=False,**tags):
        def handler_decorator(func):
            def handler(*args, **kw):
                response = func(*args, **kw)
                result = {}
                for key, tag_name in tags.items():
                    result[key] = ResponseHandler.get_xml_tag_content(response, tag_name)
                return result
            async def async_handler(*args, **kw):
                response = await func(*args, **kw)
                result = {}
                for key, tag_name in tags.items():
                    result[key] = ResponseHandler.get_xml_tag_content(response, tag_name)
                return result
            if multi:
                return async_handler
            return handler
        return handler_decorator

class ResponseChecker:
    @staticmethod
    def xml_tag_checker(tag_name,count=1):
        def checker(response):
            result = re.findall(f'<{tag_name}>(.*?)</{tag_name}>', response, re.S)
            return len(result) == count
        return checker

    @staticmethod
    def syntax_error_checker(tag_name):
        def checker(response):
            code = ResponseHandler.get_xml_tag_content(response,tag_name)
            try:
                exec(code)
                return True
            except:
                return False
        return checker

    @staticmethod
    def xml_tag_list_checker(tag_name):
        def checker(response):
            result = re.findall(f'<{tag_name}>(.*?)</{tag_name}>', response, re.S)
            return len(result) >= 1
        return checker

    @staticmethod
    def xml_content_options_checker(tag_name,options=["YES","NO"],upper=True):
        def checker(response):
            content: str = ResponseHandler.get_xml_tag_content(response,tag_name).strip()
            if upper:
                content = content.upper()
            return content in options
        return checker

    @staticmethod
    def xml_content_integer_checker(tag_name):
        def checker(response):
            content: str = ResponseHandler.get_xml_tag_content(response,tag_name)
            return content.isdigit()
        return checker

    @staticmethod
    def multi_checker(*checkers):
        def checker(response):
            for sub_checker in checkers:
                if not sub_checker(response):
                    return False
            return True
        return checker

class Linguist(BaseAgent):
    system = linguist_system
    actions_template={
        "initial":{
            "prompt":linguist_prompt,
            "keyword": ["prompt"],
        }
    }
    def __init__(self, task:Task) -> None:
        super().__init__(task)

    @BaseAgent.status_update("is_English_question")
    @ResponseHandler.assert_xml_tag_equal_handler("RESULT", "YES")
    @BaseAgent.retry(check_function=ResponseChecker.multi_checker(
        ResponseChecker.xml_tag_checker("RESULT"), ResponseChecker.xml_content_options_checker("RESULT")
    ))
    def check_English_task(self):
        question = self.task.question
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={"prompt":question})
        response = asyncio.run(session.chat(prompt))
        return response

class Translator(BaseAgent):
    system = translator_system
    actions_template={
        "initial":{
            "prompt":translator_prompt,
            "keyword": ["prompt"],
        }
    }
    def __init__(self, task: Task) -> None:
        super().__init__(task)

    @BaseAgent.status_update("question")
    @ResponseHandler.xml_tag_content_handler("PROMPT")
    @BaseAgent.retry(check_function=ResponseChecker.xml_tag_checker("PROMPT"))
    def translate_question(self):
        if self.task.status.get("is_English_question"):
            return f"<PROMPT>{self.task.question}</PROMPT>"
        question = self.task.question
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={"prompt":question})
        response = asyncio.run(session.chat(prompt))
        return response

class PromptEngineer(BaseAgent):
    system = prompt_engineer_system
    actions_template={
        "initial":{
            "prompt":prompt_engineer_prompt,
            "keyword": ["prompt"],
        }
    }
    def __init__(self, task: Task) -> None:
        super().__init__(task)

    @BaseAgent.status_update("refine_question")
    @ResponseHandler.xml_tag_content_handler("PROMPT")
    @BaseAgent.retry(check_function=ResponseChecker.xml_tag_checker("PROMPT"))
    def refine_prompt(self):
        question = self.task.status.get("question")
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={"prompt":question})
        response = asyncio.run(session.chat(prompt))
        return response

class FileAnalyst(BaseAgent):
    system = file_agent_system
    actions_template={
        "initial":{
            "prompt":file_agent_prompt,
            "keyword": ["prompt","file"],
        },
    }
    def __init__(self, task: Task) -> None:
        super().__init__(task)

    @ResponseHandler.xml_tag_content_handler("ANALYSE", multi=True)
    @BaseAgent.retry(check_function=ResponseChecker.xml_tag_checker("ANALYSE"), multi=True)
    async def analyse_file(self, file):
        question = self.task.status.get("question")
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={
            "prompt":question,
            "file":file
        })
        response = await session.chat(prompt)
        return response

    async def analyse_files(self):
        files = self.task.file_list
        tasks = []
        result = {}
        for file in files:
            tasks.append({
                "file":file,
                "task":asyncio.create_task(self.analyse_file(file))
            })
        for task in tasks:
            result[task["file"]] = await task["task"]
        self.task.status.file_analysis = result


class ToolScorer(BaseAgent):
    system = tool_scorer_system
    actions_template={
        "initial":{
            "prompt":tool_scorer_prompt,
            "keyword": ["prompt","toolname","documentation"],
        },
    }
    def __init__(self, task: Task) -> None:
        super().__init__(task)
        self.temperature = 0

    @ResponseHandler.multi_xml_tag_content_handler(multi=True,score="SCORE",reason="REASON")
    @BaseAgent.retry(check_function=ResponseChecker.multi_checker(
        ResponseChecker.xml_tag_checker("SCORE"),
        ResponseChecker.xml_tag_checker("REASON"),
        ResponseChecker.xml_content_integer_checker("SCORE"),
    ), multi=True)
    async def tool_score(self, tool):
        document = self.task.status.tools[tool].get("document")
        question = self.task.status.get("question")
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={
            "prompt":question,
            "toolname":tool,
            "documentation":document
        })
        if self.config.USE_MEMORY:
            prompt += memory_retriever.match(tool,self.task.status.raw_question, self.task.status.task_id, k=5)
        response = await session.chat(prompt)
        return response

    async def tools_score(self):
        tools = self.task.status.tools
        tasks = []
        for tool in tools:
            tasks.append({
                "tool":tool,
                "task":asyncio.create_task(self.tool_score(tool))
            })
        for task in tasks:
            response = await task["task"]
            response["score"] = int(response["score"])
            tools[task["tool"]].update(response)

class ToolDescriptor(BaseAgent):
    system = bioinformatician_system
    actions_template={
        "initial":{
            "prompt":bioinformatician_prompt,
            "keyword": ["prompt","toolname","documentation"],
        },
    }
    def __init__(self, task: Task) -> None:
        super().__init__(task)

    def get_need_describe_tools(self):
        result = []
        tools = self.task.status.tools
        key = "score" if not self.task.status.rescored else "re_score"
        for tool in tools:
            if "description" in tool: continue
            if tools[tool][key] >= self.config.HIGHSCORE_TOOL_THRESHOLD:
                result.append(tool)
        return result

    @ResponseHandler.xml_tag_content_handler("DESCRIPTION", multi=True)
    @BaseAgent.retry(check_function=ResponseChecker.xml_tag_checker("DESCRIPTION"), multi=True)
    async def tool_describe(self, tool):
        document = self.task.status.tools[tool].get("document")
        question = self.task.status.get("question")
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={
            "prompt":question,
            "toolname":tool,
            "documentation":document
        })
        response = await session.chat(prompt)
        return response

    async def tools_describe(self):
        need_describe_tools = self.get_need_describe_tools()
        tools = self.task.status.tools
        tasks = []
        for tool in need_describe_tools:
            tasks.append({
                "tool":tool,
                "task":asyncio.create_task(self.tool_describe(tool))
            })
        for task in tasks:
            response = await task["task"]
            tools[task["tool"]]["description"] = response

class ToolReScorer(BaseAgent):
    system = tool_rescorer_system
    actions_template={
        "initial":{
            "prompt":tool_rescorer_prompt,
            "keyword": ["prompt","origin_score","toolname","documentation"],
        },
    }
    def __init__(self, task: Task) -> None:
        super().__init__(task)

    def get_origin_score(self):
        tools = self.task.status.tools
        result = []
        for tool in tools:
            if tools[tool]["score"] >= self.config.HIGHSCORE_TOOL_THRESHOLD:
                result.append({
                    "toolname":tool,
                    "suggestion":tools[tool]["description"]
                })
        return json.dumps(result,ensure_ascii=False)

    @ResponseHandler.multi_xml_tag_content_handler(multi=True,re_score="SCORE",re_reason="REASON")
    @BaseAgent.retry(check_function=ResponseChecker.multi_checker(
        ResponseChecker.xml_tag_checker("SCORE"),
        ResponseChecker.xml_tag_checker("REASON"),
        ResponseChecker.xml_content_integer_checker("SCORE"),
    ), multi=True)
    async def tool_score(self, tool, origin_score):
        document = self.task.status.tools[tool].get("document")
        question = self.task.status.get("question")
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={
            "prompt":question,
            "toolname":tool,
            "documentation":document,
            "origin_score":origin_score
        })
        if self.config.USE_MEMORY:
            prompt += memory_retriever.match(tool,self.task.status.raw_question, self.task.status.task_id, k=5)
        response = await session.chat(prompt)
        return response

    async def tools_score(self):
        origin_score = self.get_origin_score()
        tools = self.task.status.tools
        tasks = []
        for tool in tools:
            tasks.append({
                "tool":tool,
                "task":asyncio.create_task(self.tool_score(tool, origin_score))
            })
        for task in tasks:
            response = await task["task"]
            response["re_score"] = int(response["re_score"])
            tools[task["tool"]].update(response)
        self.task.status.rescored = True

class WorkflowDesigner(BaseAgent):
    system = workflow_architect_system
    actions_template={
        "initial":{
            "prompt":workflow_architect_prompt,
            "keyword": ["prompt","analysis_result","tool_list"],
        },
    }
    def __init__(self, task: Task) -> None:
        super().__init__(task)
        self.model = self.config.SUPER_LLM_MODEL

    def get_tools_info(self):
        tools = self.task.status.tools
        tools_info = []
        for tool in tools:
            if tools[tool]["re_score"] >= self.config.WORKFLOW_USED_TOOL_THRESHOLD:
                tools_info.append({
                    "tool": tool,
                    "documentation": tools[tool]["document"],
                })
                if "description" in tools[tool]:
                    tools_info[-1]["description"]=tools[tool]["description"]
        return tools_info

    @BaseAgent.status_update(key="workflow")
    @ResponseHandler.xml_tag_content_handler("RESULT")
    @BaseAgent.status_update(key="raw_workflow")
    @BaseAgent.retry(check_function=ResponseChecker.xml_tag_checker("RESULT"))
    def workflow_design(self):
        tools_info = self.get_tools_info()
        question = self.task.status.get("question")
        file_analysis = self.task.status.get("file_analysis")
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={
            "prompt":question,
            "analysis_result":json.dumps(file_analysis,ensure_ascii=False),
            "tool_list":json.dumps(tools_info,ensure_ascii=False)
        })
        prompt += self.task.file_appendix
        if self.config.USE_MEMORY:
            prompt += memory_retriever.match("workflow",self.task.status.raw_question, self.task.status.task_id, k=3)
        response = asyncio.run(session.chat(prompt))
        return response

class ToolAnasyst(BaseAgent):
    system = tool_suggestion_system
    actions_template={
        "initial":{
            "prompt":tool_suggestion_prompt,
            "keyword": ["prompt","workflow","tool_info","tool"],
        },
    }
    def __init__(self, task: Task) -> None:
        super().__init__(task)
        self.model = self.config.SUPER_LLM_MODEL

    def get_tool_used(self, workflow):
        tools = self.task.status.tools
        tool_used = {}
        for tool in tools:
            if tool in workflow:
                tool_used[tool] = {
                    "tool":tool,
                    "documentation":tools[tool]["document"]
                }
        self.task.status.tool_used = tool_used
        return tool_used

    @ResponseHandler.xml_tag_content_handler("SUGGESTION", multi=True)
    @BaseAgent.retry(check_function=ResponseChecker.xml_tag_checker("SUGGESTION"), multi=True)
    async def tool_analyse(self, tool, workflow):
        question = self.task.status.get("question")
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={
            "prompt":question,
            "tool":tool,
            "workflow":workflow,
            "tool_info":self.tool_info
        })
        response = await session.chat(prompt)
        return response

    async def tools_analyse(self):
        workflow = self.task.status.get("workflow")
        tools = self.get_tool_used(workflow)
        self.tool_info = json.dumps(tools, ensure_ascii=False)
        tasks = []
        for tool in tools:
            tasks.append({
                "tool":tool,
                "task":asyncio.create_task(self.tool_analyse(tool, workflow))
            })
        for task in tasks:
            response = await task["task"]
            tools[task["tool"]]["suggestion"] = response

class WorkflowFormatter(BaseAgent):
    system = format_desinger_system
    actions_template={
        "initial":{
            "prompt":format_desinger_prompt,
            "keyword": ["prompt","workflow"],
        },
    }
    def __init__(self, task: Task) -> None:
        super().__init__(task)

    @BaseAgent.status_update("workflow_stages")
    @ResponseHandler.xml_tag_list_content_handler("STAGE")
    @BaseAgent.retry(check_function=ResponseChecker.xml_tag_list_checker("STAGE"))
    def workflow_format(self):
        question = self.task.status.get("question")
        workflow = self.task.status.get("workflow")
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={
            "prompt":question,
            "workflow":workflow
        })
        response = asyncio.run(session.chat(prompt))
        return response

class ActionDesigner(BaseAgent):
    system = action_architecture_expert_system
    actions_template={
        "initial":{
            "prompt":action_architecture_expert_prompt,
            "keyword": ["prompt","workflow","tool_suggestion_data","stage"],
        },
    }
    def __init__(self, task: Task) -> None:
        super().__init__(task)
        self.model = self.config.SUPER_LLM_MODEL

    @ResponseHandler.xml_tag_content_handler("STAGE", multi=True)
    @BaseAgent.retry(check_function=ResponseChecker.xml_tag_checker("STAGE"), multi=True)
    async def action_design(self, stage):
        question = self.task.status.get("question")
        workflow = self.task.status.get("workflow")
        tool_suggestion_data = json.dumps(self.task.status.tool_used, ensure_ascii=False)
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={
            "prompt":question,
            "stage":stage,
            "workflow":workflow,
            "tool_suggestion_data":tool_suggestion_data
        })
        response = await session.chat(prompt)
        return response

    async def actions_design(self):
        stages = self.task.status.get("workflow_stages")
        tasks = []
        actions = []
        for stage in stages:
            tasks.append({
                "stage":stage,
            })

        for task in tasks:
            actions.append({
                "stage":task["stage"],
            })
        self.task.status.actions = actions

class MermaidDesigner(BaseAgent):
    system = mermaid_system
    actions_template={
        "initial":{
            "prompt":mermaid_prompt,
            "keyword": ["workflow"],
        },
    }
    def __init__(self, task: Task) -> None:
        super().__init__(task)
        self.model = self.config.SUPER_LLM_MODEL

    @BaseAgent.status_update("mermaid_code")
    @ResponseHandler.xml_tag_content_handler("CODE")
    @BaseAgent.retry(check_function=ResponseChecker.xml_tag_checker("CODE"))
    def mermaid_design(self):
        workflow = self.task.status.get("workflow")
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={
            "workflow":workflow
        })
        response = asyncio.run(session.chat(prompt))
        return response

class Programmer(BaseAgent):
    programmer_system = programmer_system
    tester_system = tester_system
    programmer_template = {
        "initial":{
            "prompt":programmer_prompt,
            "keyword": ["prompt","workflow","tool_suggestion_data","subtask","name"],
        },
        "func_fix":{
            "prompt":programmer_func_fix,
            "keyword": ["function"]
        },
        "fail_test":{
            "prompt":programmer_fail_test,
            "keyword": ["suggestion"]
        }
    }
    tester_template = {
        "initial":{
            "prompt":tester_prompt,
            "keyword": ["prompt","workflow","tool_suggestion_data","subtask","code","function","resource_pool"],
        },
        "rethink":{
            "prompt":tester_rethink,
            "keyword": ["info"]
        },
        "error":{
            "prompt":tester_error,
            "keyword": ["info"]
        },
        "description":{
            "prompt":tester_description,
            "keyword": ["output"]
        },
        "fail_test":{
            "prompt":tester_fail_test,
            "keyword": ["suggestion"]
        },
        "fix":{
            "prompt":tester_fix,
            "keyword": []
        },
    }

    necessary_system = necessary_system
    necessary_template={
        "initial":{
            "prompt":necessary_prompt,
            "keyword": ["prompt","workflow","subtask"],
        },
    }

    split_system = split_again_system
    split_template = {
        "initial":{
            "prompt":split_again_prompt,
            "keyword": ["prompt","workflow","subtask"],
        },
    }

    workflow_fix_system = workflow_fix_system
    workflow_fix_template = {
        "ignore":{
            "prompt":workflow_fix_ignore,
            "keyword":["prompt","workflow","subtask","error"]
        },
        "split":{
            "prompt":workflow_fix_split,
            "keyword":["prompt","workflow","subtask","error"]
        },
        "normal":{
            "prompt":workflow_fix_normal,
            "keyword":["prompt","workflow","subtask","error"]
        }
    }

    def __init__(self, task: Task) -> None:
        super().__init__(task)
        self.model = self.config.SUPER_LLM_MODEL

        self.programmer_extra_prompt = ""
        self.tester_extra_prompt = ""

    def get_tool_suggestion_data(self):
        tools = self.task.status.tool_used
        result = []
        for tool in tools:
            result.append({
                "tool":tool,
                "documentation":tools[tool]["documentation"]
            })
        return json.dumps(result,ensure_ascii=False)

    @ResponseHandler.xml_tag_content_handler("CODE")
    @BaseAgent.retry(check_function=ResponseChecker.multi_checker(
        ResponseChecker.xml_tag_checker("CODE")
    ))
    def get_programmer_code(self, subtask, name, session):
        question = self.task.status.get("question")
        workflow = self.task.status.get("workflow")
        tool_suggestion_data = self.get_tool_suggestion_data()
        prompt = self.format_template(data={
            "prompt":question,
            "workflow":workflow,
            "subtask":subtask,
            "tool_suggestion_data":tool_suggestion_data,
            "name":name
        },templates=self.programmer_template)
        if name.endswith("0"):
            prompt += self.task.file_appendix
        if self.config.USE_MEMORY:
            prompt += memory_retriever.match("action",subtask, self.task.status.task_id, k=5)
        response = asyncio.run(session.chat(prompt))
        return response

    @ResponseHandler.xml_tag_content_handler("CODE")
    @BaseAgent.retry(check_function=ResponseChecker.multi_checker(
        ResponseChecker.xml_tag_checker("CODE")
    ))
    def get_tester_code(self, subtask, code, function, session):
        question = self.task.status.get("question")
        workflow = self.task.status.get("workflow")
        tool_suggestion_data = self.get_tool_suggestion_data()
        resource_pool = json.dumps(self.task.status.resource_pool, ensure_ascii=False)
        prompt = self.format_template(data={
            "prompt":question,
            "workflow":workflow,
            "subtask":subtask,
            "tool_suggestion_data":tool_suggestion_data,
            "function":function,
            "code":code,
            "resource_pool":resource_pool,
        }, templates=self.tester_template)
        if function.endswith("0"):
            prompt += self.task.file_appendix
        if self.config.USE_MEMORY:
            prompt += memory_retriever.match("test",subtask, self.task.status.task_id, k=5)
        response = asyncio.run(session.chat(prompt))
        return response

    @ResponseHandler.xml_tag_content_handler("SUGGESTION")
    @BaseAgent.retry(check_function=ResponseChecker.xml_tag_checker("SUGGESTION"))
    def get_fix_suggestion(self, subtask, error, chat_action):
        question = self.task.status.get("question")
        workflow = self.task.status.get("workflow")
        prompt = self.format_template(action=chat_action,data={
            "prompt":question,
            "workflow":workflow,
            "subtask":subtask,
            "error":error
        },templates=self.workflow_fix_template)
        session = self.create_chatsession(self.workflow_fix_system)
        response = asyncio.run(session.chat(prompt))
        return response

    @ResponseHandler.xml_tag_content_handler("CODE")
    @BaseAgent.retry(check_function=ResponseChecker.multi_checker(
        ResponseChecker.xml_tag_checker("CODE")
    ))
    def fix_programming_code(self, suggestion, session:ChatSession):
        prompt = self.format_template(action="fail_test",data={
            "suggestion":suggestion
        }, templates=self.tester_template)
        response = asyncio.run(session.chat(prompt))
        return response

    @ResponseHandler.xml_tag_content_handler("CODE")
    @BaseAgent.retry(check_function=ResponseChecker.multi_checker(
        ResponseChecker.xml_tag_checker("CODE")
    ))
    def fix_test_code(self, session:ChatSession):
        prompt = self.format_template(action="fix",data={}, templates=self.tester_template)
        response = asyncio.run(session.chat(prompt))
        return response


    @ResponseHandler.multi_xml_tag_content_handler(reason="REASON")
    @BaseAgent.retry(check_function=ResponseChecker.multi_checker(
        ResponseChecker.xml_tag_checker("REASON")
    ))
    def get_reason(self, info, chat_action, session:ChatSession):
        prompt = self.format_template(action=chat_action,data={
            "info":info
        }, templates=self.tester_template)
        response = asyncio.run(session.chat(prompt))
        return response

    @ResponseHandler.multi_xml_tag_content_handler(description="DESCRIPTION",output="OUTPUT",type="TYPE")
    @BaseAgent.retry(check_function=ResponseChecker.multi_checker(
        ResponseChecker.xml_tag_checker("DESCRIPTION"),
        ResponseChecker.xml_tag_checker("OUTPUT"),
        ResponseChecker.xml_tag_checker("TYPE")
    ))
    def resource_analyse(self, data, session=None):
        if session is None:
            session = self.create_chatsession("You are a tester")
        prompt = self.format_template(action="description",data={
            "output":json.dumps(data)
        }, templates=self.tester_template)
        response = asyncio.run(session.chat(prompt))
        return response

    @ResponseHandler.assert_xml_tag_equal_handler("RESULT", "NO")
    @BaseAgent.retry(check_function=ResponseChecker.multi_checker(
        ResponseChecker.xml_tag_checker("RESULT"), ResponseChecker.xml_content_options_checker("RESULT")
    ))
    def check_can_ignore(self, subtask):
        question = self.task.status.get("question")
        workflow = self.task.status.get("workflow")
        prompt = self.format_template(data={
            "prompt":question,
            "workflow":workflow,
            "subtask":subtask
        },templates=self.necessary_template)
        session = self.create_chatsession(self.necessary_system)
        response = asyncio.run(session.chat(prompt))
        return response


    @ResponseHandler.assert_xml_tag_equal_handler("RESULT", "YES")
    @BaseAgent.retry(check_function=ResponseChecker.multi_checker(
        ResponseChecker.xml_tag_checker("RESULT"), ResponseChecker.xml_content_options_checker("RESULT")
    ))
    def check_can_split(self, subtask):
        question = self.task.status.get("question")
        workflow = self.task.status.get("workflow")
        prompt = self.format_template(data={
            "prompt":question,
            "workflow":workflow,
            "subtask":subtask
        },templates=self.split_template)
        session = self.create_chatsession(self.split_system)
        response = asyncio.run(session.chat(prompt))
        return response

    def action_execute(self, code, test_func_name):
        code_task_id = push_code(code, test_func_name, self.config, official=self.task.official)
        output = get_code_output(code_task_id, self.config)
        return output

    def action_task(self, action, index):
        programmer_session = self.create_chatsession(self.programmer_system)
        programmer_code = self.get_programmer_code(action, f"action{index}", programmer_session)
        tester_session = self.create_chatsession(self.tester_system)
        tester_code = self.get_tester_code(action, programmer_code, f"test{index}", tester_session)
        action_code = get_action_code(programmer_code, tester_code)
        result, data = self.action_task_posting(
            action,
            programmer_code,
            tester_code,
            index,
            programmer_session,
            tester_session
        )
        if result:
            #? ######################################################
            self.task.status.code[f"stage{index}"] = action_code
            self.task.status.code_result["partial_result"].append({
                "action": index,
                "finish": True,
                "code": action_code,
                "files":[], # TODO
                "path":self.config.TASK_DIR
            })
            self.task.status.status_update("code")
            #? ######################################################
            return True, action_code
        retry_times = 1
        while retry_times < self.config.ACTION_RETRY_TIMES:
            #? ######################################################
            self.task.status.code_result["partial_result"].append({
                "action": index,
                "finish": False,
                "code": action_code,
                "retry": retry_times,
                "files":[],#TODO
                "path":self.config.TASK_DIR
            })
            self.task.status.status_update("code")
            #? ######################################################
            result, data = self.action_task_posting(
                action,
                data["programmer_code"],
                data["tester_code"],
                index,
                data["programmer_session"],
                data["tester_session"]
            )
            action_code = get_action_code(data["programmer_code"], data["tester_code"])
            if result:
                self.task.status.code[f"stage{index}"] = action_code
                self.task.status.code_result["partial_result"].append({
                    "action": index,
                    "finish": True,
                    "code": action_code,
                    "files":[],#TODO
                    "path":self.config.TASK_DIR
                })
                self.task.status.status_update("code")
                return True, action_code
            retry_times += 1
        self.task.status.code_result["partial_result"].append({
            "action": index,
            "finish": False,
            "code": action_code,
            "retry": retry_times,
            "files":[],
            "path":self.config.TASK_DIR,
        })
        self.task.status.status_update("code")
        can_ignore = self.check_can_ignore(action)
        can_split = self.check_can_split(action)
        can_split = False
        if can_ignore:
            chat_action = "ignore"
        elif can_split:
            chat_action = "split"
        else:
            chat_action = "normal"
        suggestion = self.get_fix_suggestion(action,data["error_reason"], chat_action)
        return False, suggestion

    def action_task_posting(self, action, programmer_code, tester_code, index, programmer_session, tester_session):
        action_code = get_action_code(programmer_code, tester_code)
        output = self.action_execute(action_code, f"test{index}")
        result = output["result"]
        excepted = output["excepted"]
        data = output["data"]
        if not result:
            chat_action = "error" if excepted else "rethink"
            response = self.get_reason(data, chat_action, tester_session)
            test_result = False
            test_reason = response["reason"]
            if test_result:
                tester_code = self.fix_test_code(tester_session)
            else:
                programmer_code = self.fix_programming_code(test_reason, programmer_session)
                tester_session = self.create_chatsession(self.tester_system)
                tester_code = self.get_tester_code(action, programmer_code, f"test{index}", tester_session)
            return False, {
                "programmer_code": programmer_code,
                "tester_code": tester_code,
                "programmer_session": programmer_session,
                "tester_session": tester_session,
                "output":output,
                "error_reason":test_reason,
            }
        else:
            response = self.resource_analyse(data, tester_session)
            description = response["description"]
            output = response["output"]
            _type = response["type"]
            self.task.status.resource_pool.append({
                "item":data,
                "info":output,
                "type":_type,
                "description":description
            })
            return True, {
                "programmer_code": programmer_code,
                "tester_code": tester_code,
            }

    def programming(self):
        self.task.status.resource_pool = []
        self.task.status.code_result = {
            "partial_result" : [],
        }
        total_result = {
            "code_info":[],
            "path":self.config.TASK_DIR,
            "files":[]
        }
        for k,v in self.task.status.file_analysis.items():
            self.task.status.resource_pool.append({
                "item":k,
                "info":k,
                "type":"file",
                "description":v
            })
        actions = self.task.status.get("actions")
        for index,action in enumerate(actions):
            ok, data = self.action_task(action["stage"], index)
            total_result["code_info"].append({
                "task":action["stage"],
                "code":data
            })
            if not ok:
                return False, data
        self.task.status.code_result["total_result"] = total_result
        self.task.status.status_update("code")
        return True, None

class WorkflowReDesigner(BaseAgent):
    system = re_workflow_system
    actions_template={
        "initial":{
            "prompt":re_workflow_prompt,
            "keyword": ["prompt","analysis_result","tool_list","workflow","suggestion"],
        },
    }
    def __init__(self, task: Task) -> None:
        super().__init__(task)
        self.model = self.config.SUPER_LLM_MODEL

    def get_tools_info(self):
        tools = self.task.status.tools
        tools_info = []
        for tool in tools:
            if tools[tool]["re_score"] >= self.config.WORKFLOW_USED_TOOL_THRESHOLD:
                tools_info.append({
                    "tool": tool,
                    "documentation": tools[tool]["document"],
                })
                if "description" in tools[tool]:
                    tools_info[-1]["description"]=tools[tool]["description"]
        return tools_info

    @BaseAgent.status_update(key="workflow")
    @ResponseHandler.xml_tag_content_handler("RESULT")
    @BaseAgent.status_update(key="raw_workflow")
    @BaseAgent.retry(check_function=ResponseChecker.xml_tag_checker("RESULT"))
    def workflow_redesign(self, suggestion):
        tools_info = self.get_tools_info()
        question = self.task.status.get("question")
        file_analysis = self.task.status.get("file_analysis")
        workflow = self.task.status.get("raw_workflow")
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={
            "prompt":question,
            "workflow":workflow,
            "suggestion":suggestion,
            "analysis_result":json.dumps(file_analysis,ensure_ascii=False),
            "tool_list":json.dumps(tools_info,ensure_ascii=False)
        })
        if self.config.USE_MEMORY:
            prompt += memory_retriever.match("workflow",self.task.status.raw_question, self.task.status.task_id, k=3)
        response = asyncio.run(session.chat(prompt))
        return response

class SummaryAnalyst(BaseAgent):
    system = summary_system
    actions_template={
        "initial":{
            "prompt":summary_prompt,
            "keyword": ["prompt","workflow","resource_pool"],
        },
    }
    def __init__(self, task: Task) -> None:
        super().__init__(task)
        self.model = self.config.SUPER_LLM_MODEL

    @BaseAgent.status_update("summary")
    @ResponseHandler.xml_tag_content_handler("SUMMARY")
    @BaseAgent.retry(check_function=ResponseChecker.xml_tag_checker("SUMMARY"))
    def summary(self):
        question = self.task.status.get("question")
        workflow = self.task.status.get("workflow")
        resource_pool = json.dumps(self.task.status.resource_pool, ensure_ascii=False)

        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={
            "prompt":question,
            "workflow":workflow,
            "resource_pool":resource_pool
        })
        response = asyncio.run(session.chat(prompt))
        return response

# Cell number 14

# scripts -> bot.py

import asyncio

#from agent import BaseAgent, ResponseHandler, ResponseChecker
#from config import Config

'''
from scripts.prompt import (
    request_analyse_prompt,
    request_analyse_system,
    workflow_analyse_prompt,
    workflow_analyse_system,
    tool_suggestion_system,
    tool_suggestion_prompt,
    mermaid_system,
    mermaid_prompt
)
'''

class RequestAnalyst(BaseAgent):
    system = request_analyse_system
    actions_template = {
        "initial":{
            "prompt":request_analyse_prompt,
            "keyword": [
                "tool_name",
                "tool_doc",
                "tool_code",
                "data_info"
            ],
        }
    }

    def __init__(self) -> None:
        self.config = Config()
        self.config.SAVE_LOG = False
        self.temperature = 0.2
        self.model = self.config.SUPER_LLM_MODEL

    @ResponseHandler.xml_tag_content_handler("PROMPT")
    @BaseAgent.retry(check_function=ResponseChecker.xml_tag_checker("PROMPT"))
    def request_analyse(self, name, doc, code, info):
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={
            "tool_name":name,
            "tool_doc":doc,
            "tool_code":code,
            "data_info":info,
        })
        response = asyncio.run(session.chat(prompt))
        return response

class WorkflowAnalyst(BaseAgent):
    system = workflow_analyse_system
    actions_template={
        "initial":{
            "prompt":workflow_analyse_prompt,
            "keyword": [
                "question",
                "tool_name",
                "tool_doc",
                "action_test_code"
                ],
        },
    }
    def __init__(self) -> None:
        self.config = Config()
        self.config.SAVE_LOG = False
        self.temperature = 0.2
        self.model = self.config.SUPER_LLM_MODEL

    @ResponseHandler.xml_tag_content_handler("RESULT")
    @BaseAgent.retry(check_function=ResponseChecker.xml_tag_checker("RESULT"))
    def workflow_analyse(self, question, tool_name, tool_doc, action_test_code):
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={
            "question": question,
            "tool_name": tool_name,
            "tool_doc": tool_doc,
            "action_test_code": action_test_code
        })
        return asyncio.run(session.chat(prompt))

class SuggestionAnalyst(BaseAgent):
    system = tool_suggestion_system
    actions_template={
        "initial":{
            "prompt":tool_suggestion_prompt,
            "keyword": [
                "prompt",
                "tool",
                "workflow",
                "tool_info"
            ]
        },
    }
    def __init__(self) -> None:
        self.config = Config()
        self.config.SAVE_LOG = False
        self.temperature = 0.2
        self.model = self.config.SUPER_LLM_MODEL

    @ResponseHandler.xml_tag_content_handler("SUGGESTION")
    @BaseAgent.retry(check_function=ResponseChecker.xml_tag_checker("SUGGESTION"))
    def tool_suggestion_analyse(self, question, tool, workflow, tool_info):
        # question = self.task.status.get("question")
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={
            "prompt":question,
            "tool":tool,
            "workflow":workflow,
            "tool_info":tool_info
        })
        response = asyncio.run(session.chat(prompt))
        return response

class MermaidDesigner(BaseAgent):
    system = mermaid_system
    actions_template={
        "initial":{
            "prompt":mermaid_prompt,
            "keyword": ["workflow"],
        },
    }
    def __init__(self) -> None:
        self.config = Config()
        self.config.SAVE_LOG = False
        self.temperature = 0.2
        self.model = self.config.SUPER_LLM_MODEL

    @ResponseHandler.xml_tag_content_handler("CODE")
    @BaseAgent.retry(check_function=ResponseChecker.xml_tag_checker("CODE"))
    def mermaid_design(self, workflow):
        session = self.create_chatsession(self.system)
        prompt = self.format_template(data={
            "workflow":workflow
        })
        response = asyncio.run(session.chat(prompt))
        return response


# Cell number 15

# memory_retriever.py

import json
import time
#from config import Config
#from utils import generate_task_id

class TestMemoryRetriever:
    def __init__(self) -> None:
        self.r = Config().get_redis_connect()
        self.config = Config()

    def match(self, item_key, query_key, total_task_id, k=2):
        k = 3
        task_id = generate_task_id()
        data = {
            "task_id":task_id,
            "item_key":item_key,
            "query_key":query_key,
            "k":k,
            "threshold":0.4
        }
        self.r.lpush(self.config.REDIS_MEMORY_TASK_KEY,json.dumps(data))
        while True:
            result = self.r.hget(self.config.REDIS_MEMORY_RESULT_KEY, task_id)
            if result is None:
                time.sleep(1)
            else:
                result = json.loads(result)
                if len(result) == 0:
                    self.config.log_error("special", "result is None")
                    return ""
                content = []
                for item in result:
                    content.append(item["value"])
                    self.r.lpush(self.config.REDIS_MEMORY_LOG2, json.dumps({
                        "item_key":item_key,
                        "current_query_key":query_key,
                        "origin_query_key":item["key"],
                        "query_value":item["value"],
                        "memory_id":item["id"],
                        "origin_question_id":item["question_id"],
                        "current_task_id":total_task_id,
                        "k":len(result),
                        "score":item["score"]
                    }))
                content = f"\n{'='*10}\n".join(content)
                response = f"""
Here's a summary of some of your experiences and memories that you can use for reference: 【\n{content}\n】You must learn from experience and memory as well as reference!
"""
                return response

# Cell number 16+17 merged

import subprocess
from pathlib import Path

subprocess.run(
    ["pip", "-q", "install", "pandas", "scipy"],
    check=True
)

patch_code = r'''
import pandas as pd
from scipy.stats import ttest_ind

def _read_table(path):
    return pd.read_csv(path, sep=None, engine="python")

def t_test(data_file, group_file, groupA_name, groupB_name, id, group):
    data_df = _read_table(data_file)
    group_df = _read_table(group_file)

    first_col = data_df.columns[0]
    data_df = data_df.set_index(first_col)

    data_df.columns = data_df.columns.map(str)
    group_df[id] = group_df[id].map(str)
    group_df[group] = group_df[group].map(str)

    groupA_samples = group_df.loc[group_df[group] == str(groupA_name), id].tolist()
    groupB_samples = group_df.loc[group_df[group] == str(groupB_name), id].tolist()

    groupA_samples = [s for s in groupA_samples if s in data_df.columns]
    groupB_samples = [s for s in groupB_samples if s in data_df.columns]

    if not groupA_samples:
        raise ValueError(f"No samples found for groupA={groupA_name!r}")
    if not groupB_samples:
        raise ValueError(f"No samples found for groupB={groupB_name!r}")

    rows = []
    for feature, row in data_df.iterrows():
        a = pd.to_numeric(row[groupA_samples], errors="coerce").dropna().astype(float)
        b = pd.to_numeric(row[groupB_samples], errors="coerce").dropna().astype(float)

        if len(a) >= 2 and len(b) >= 2:
            t_stat, p_value = ttest_ind(a, b, equal_var=True, nan_policy="omit")
        else:
            t_stat, p_value = float("nan"), float("nan")

        rows.append({
            "biomarker": feature,
            "feature": feature,
            "groupA": str(groupA_name),
            "groupB": str(groupB_name),
            "groupA_n": int(len(a)),
            "groupB_n": int(len(b)),
            "groupA_mean": float(a.mean()) if len(a) else float("nan"),
            "groupB_mean": float(b.mean()) if len(b) else float("nan"),
            "groupA_std": float(a.std(ddof=1)) if len(a) > 1 else float("nan"),
            "groupB_std": float(b.std(ddof=1)) if len(b) > 1 else float("nan"),
            "t_stat": float(t_stat) if pd.notna(t_stat) else float("nan"),
            "p_value": float(p_value) if pd.notna(p_value) else float("nan"),
        })

    out_df = pd.DataFrame(rows).sort_values(
        by=["p_value", "biomarker"], na_position="last"
    )

    out_file = "out_t_test.tsv"
    out_df.to_csv(out_file, sep="\t", index=False)
    return out_file
'''

tool_path = Path("/content/BioMedAgent/tool/code/t_test")
tool_path.parent.mkdir(parents=True, exist_ok=True)
tool_path.write_text(patch_code, encoding="utf-8")

print("Patched:", tool_path)
print("Now outputs both 'biomarker' and 'feature'.")

# Cell number 18

# server -> base_server.py

import json
import time
from uuid import uuid4
from typing import Any

import redis


class BaseServer:
    CENTER_KEY_PREFIX = "biomedagent:server_center"
    HEARTBEAT_KEY = "biomedagent:server_heartbeat"
    DEAD_POOL = "biomedagent:server_dead"

    SERVER_KEY = "base"
    SLEEP_TIME = 1

    def __init__(self, host="localhost", port=6379, db=0) -> None:
        self.r = redis.Redis(host=host, port=port, db=db, decode_responses=True)
        self.center = f"{self.CENTER_KEY_PREFIX}:{self.SERVER_KEY}"
        self.id = str(uuid4())

        self._register()

    def _register(self) -> None:
        self.r.hset(
            self.center,
            self.id,
            json.dumps({
                "id": self.id,
                "status": "wait"   # ['wait', 'run']
            })
        )
        self.r.hset(self.HEARTBEAT_KEY, self.id, int(time.time()))

    def _change_status(self, status: str) -> None:
        info = self.r.hget(self.center, self.id)
        if info is None:
            return
        info = json.loads(info)
        info["status"] = status
        self.r.hset(self.center, self.id, json.dumps(info))

    def start_task(self) -> None:
        self._change_status("run")

    def finish_task(self) -> None:
        self._change_status("wait")

    def check_alive(self) -> bool:
        return not bool(self.r.hget(self.DEAD_POOL, self.id))

    def get_task(self):
        """
        Should return: (ok: bool, data: Any)
        Example:
            return False, None
            return True, {"task_id": "...", ...}
        """
        return False, None

    def execute(self, data: Any):
        """
        Override in subclasses.
        """
        raise NotImplementedError("Subclasses must implement execute().")

    def heartbeat(self) -> None:
        self.r.hset(self.HEARTBEAT_KEY, self.id, int(time.time()))

    def on_error(self, e: Exception, data: Any):
        raise e

    def run(self):
        while True:
            if not self.check_alive():
                break

            self.heartbeat()
            ok, data = self.get_task()

            if not ok:
                time.sleep(self.SLEEP_TIME)
                continue

            self.start_task()
            try:
                self.execute(data)
            except Exception as e:
                self.on_error(e, data)
            finally:
                self.finish_task()


# Cell number 19

# server -> batch_task_executor.py

import json
import time
import traceback
import asyncio

# assumes the following are already defined in earlier notebook cells:
# BaseServer, Config, Task
# Linguist, Translator, PromptEngineer, FileAnalyst
# ToolScorer, ToolDescriptor, ToolReScorer
# WorkflowDesigner, ToolAnasyst, WorkflowFormatter
# ActionDesigner, Programmer, SummaryAnalyst
# WorkflowReDesigner, MermaidDesigner
# rprint

class BatchTaskExecutor(BaseServer):
    SERVER_KEY = "batchtask-v0.2"

    def __init__(self, host="localhost", port=6379, db=0) -> None:
        super().__init__(host=host, port=port, db=db)

    def get_task(self):
        task_id = self.r.rpop("biomedagent:pool:batch_task:0.2")
        if not task_id:
            return False, None

        data = self.r.get(f"biomedagent:info:batch_task:{task_id}")
        if not data:
            return False, None

        data = json.loads(data)
        data["status"] = "start"
        self.save_data(task_id, data)
        return True, data

    def save_data(self, task_id, data):
        self.r.set(f"biomedagent:info:batch_task:{task_id}", json.dumps(data))

    def execute(self, data):
        task_id = data["task_id"]
        print(task_id)

        update = bool(data.get("update"))
        is_batch = bool(data.get("batch"))
        rprint(str(is_batch))

        start_time = time.time()

        config = Config.set_task(task_id)
        config.ECHO_INFO = False
        config.USE_MEMORY = False
        config.USE_FILE_APPENDIX = False

        task = Task(data, config, update, official=is_batch)
        task.status.raw_question = task.question

        linguist = Linguist(task)
        translator = Translator(task)
        prompt_engineer = PromptEngineer(task)
        file_analyst = FileAnalyst(task)
        tool_scorer = ToolScorer(task)
        tool_descriptor = ToolDescriptor(task)
        tool_rescorer = ToolReScorer(task)
        workflow_designer = WorkflowDesigner(task)
        tool_analyst = ToolAnasyst(task)
        workflow_formatter = WorkflowFormatter(task)
        action_designer = ActionDesigner(task)
        programmer = Programmer(task)
        summary_analyst = SummaryAnalyst(task)
        workflow_redesigner = WorkflowReDesigner(task)
        mermaid_designer = MermaidDesigner(task)

        try:
            error = "no error"
            task.status.status_update("start")

            linguist.check_English_task()
            translator.translate_question()
            prompt_engineer.refine_prompt()
            task.status.status_update("improve_prompt")

            asyncio.run(file_analyst.analyse_files())
            asyncio.run(tool_scorer.tools_score())
            asyncio.run(tool_descriptor.tools_describe())
            asyncio.run(tool_rescorer.tools_score())
            asyncio.run(tool_descriptor.tools_describe())
            task.status.status_update("tool_score")

            workflow_designer.workflow_design()
            # mermaid_designer.mermaid_design()

            asyncio.run(tool_analyst.tools_analyse())
            workflow_formatter.workflow_format()
            asyncio.run(action_designer.actions_design())
            task.status.status_update("workflow")

            success, suggestion = programmer.programming()

            retry_times = 1
            while not success and retry_times < 3:
                if retry_times >= 3:
                    programmer.tester_extra_prompt = """
You'll want to simulate the output via the mock technique and try to write code that generates a virtual empty file, it's not necessary to actually accomplish the actual function.
"""
                retry_times += 1
                task.backtrack_update()
                workflow_redesigner.workflow_redesign(suggestion)
                asyncio.run(tool_analyst.tools_analyse())
                workflow_formatter.workflow_format()
                asyncio.run(action_designer.actions_design())
                task.status.status_update("workflow")
                success, suggestion = programmer.programming()

            if not success:
                raise Exception("任务失败")

            summary_analyst.summary()
            task.status.status_update("summary")
            time.sleep(3)
            task.status.status_update("completed")

        except Exception as e:
            traceback.print_exc()
            success = False
            error = repr(e)
            task.status.error = error
            task.status.status_update("failed")

        task.status.success = success
        task.status.use_time = int(time.time() - start_time)

        data["status"] = success
        data["finish"] = True
        self.save_data(task_id, data)
        self.r.set(f"biomedagent:result:{task_id}", json.dumps(task.status.data))


# Cell number 20

# server -> code_executor.py

import os
import json

# assumes these are already defined in earlier notebook cells:
# BaseServer, Config, ToolManager, rprint, gprint

class CodeExecutor(BaseServer):
    SERVER_KEY = "code"

    def __init__(self, host="localhost", port=6379, db=0) -> None:
        super().__init__(host=host, port=port, db=db)
        self.config = Config()
        self.tool_manager = ToolManager(self.config)

    def get_task(self):
        result = self.r.rpop(self.config.REDIS_EXECUTOR_LIST_TASK_KEY)
        if result is None:
            return False, None
        return True, json.loads(result)

    def add_chdir_code(self, code, task_path):
        return f'''
import os
os.chdir(r"{task_path}")

{code}
'''

    def add_official_tools_code(self, code):
        tools_code = ""
        tools = os.listdir(self.config.TOOL_CODE_DIR)
        for tool in tools:
            if tool.startswith("."):
                continue
            if tool not in code:
                continue
            with open(os.path.join(self.config.TOOL_CODE_DIR, tool), "r", encoding="utf8") as f:
                tools_code += f"\n{f.read()}\n"
        return f"""
{tools_code}

{code}
"""

    def add_tools_code(self, code, ensure_active):
        tools_code = ""
        tools = self.tool_manager.get_tool_list(ensure_active)
        for tool in tools:
            if ensure_active:
                tool_name = tool
            else:
                tool_name = self.tool_manager.get_tool_name_by_id(tool)

            if tool_name in code:
                if ensure_active:
                    tool_code = self.tool_manager.get_tool_code(tool_name)
                else:
                    tool_code = self.tool_manager.get_tool_code_by_id(tool)

                tools_code += f"\n{tool_code}\n"

        return f"""
{tools_code}

{code}
"""

    def execute(self, data):
        task_id = data.get("task_id")
        code = data.get("code")
        test_func_name = data.get("test_func_name")
        ensure_active = data.get("ensure_active")
        task_path = data.get("task_path")
        official = data.get("official")

        code = self.add_chdir_code(code, task_path)

        # keep the original behavior from their file
        if True:
            code = self.add_official_tools_code(code)
        if False:   # TODO: dynamic tool injection path
            code = self.add_tools_code(code, ensure_active)

        os.chdir(task_path)
        namespace = {test_func_name: None}
        excepted = False

        gprint("start task")
        try:
            exec(code, namespace)
            test_func = namespace[test_func_name]
            result, result_data = test_func()

            # must be JSON serializable, same as original logic
            json.dumps(result_data)

        except TypeError as e:
            excepted = True
            result, result_data = False, repr(e)
            if "not JSON serializable" in result_data:
                result_data = "The return value of a function should be a base type and not a non-jsonizable object."

        except SyntaxError as e:
            excepted = True
            result, result_data = False, repr(e)
            result_data += "\nPlease note, double check the code for syntax issues, variable names, indentation, etc."

        except ModuleNotFoundError as e:
            excepted = True
            result, result_data = False, repr(e)
            result_data += "\nImporting hypothetical third-party libraries on your own is prohibited, while the given tool code does not need to be imported!"

        except Exception as e:
            excepted = True
            result, result_data = False, repr(e)

        rprint("finish task")

        output = {
            "excepted": excepted,
            "result": result,
            "data": result_data
        }

        self.r.hset(
            self.config.REDIS_EXECUTOR_LIST_DATA_KEY,
            task_id,
            json.dumps(output, ensure_ascii=False)
        )


# Cell number 21

# server -> gpt_server.py

import os
import json
import requests
import traceback
import warnings

from requests.exceptions import ConnectionError

# assumes these are already defined in earlier notebook cells:
# BaseServer, Config

class GPTResponseFormatError(Exception):
    pass


class GPTServer(BaseServer):
    SERVER_KEY = "gpt"

    def __init__(self, host="localhost", port=6379, db=0) -> None:
        super().__init__(host=host, port=port, db=db)
        self.config = Config()
        self.url = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1/chat/completions")

        self.headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {os.environ.get('OPENAI_API_KEY')}"
        }

    def get_task(self):
        data = self.r.rpop(self.config.REDIS_GPT_TASK_KEY)
        if not data:
            return False, None
        return True, json.loads(data)

    def on_error(self, e: Exception, data):
        self.config.log_error(self.SERVER_KEY, "".join(traceback.format_exception(e)))
        err_text = "".join(traceback.format_exception(e))

        if "string_above_max_length" in err_text:
            return
        elif "context_length_exceeded" in err_text:
            return

        if isinstance(e, requests.exceptions.ProxyError):
            self.r.rpush(self.config.REDIS_GPT_TASK_KEY, json.dumps(data))
        elif isinstance(e, GPTResponseFormatError):
            self.r.rpush(self.config.REDIS_GPT_TASK_KEY, json.dumps(data))
        elif isinstance(e, ConnectionError):
            self.r.rpush(self.config.REDIS_GPT_TASK_KEY, json.dumps(data))

    def execute(self, data: dict):
        task_id = data.get("task_id")

        post_data = data.copy()
        post_data.pop("task_id", None)

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            response = requests.post(
                self.url,
                headers=self.headers,
                json=post_data,
                verify=False
            ).json()

        if "choices" not in response:
            raise GPTResponseFormatError(json.dumps(response))

        content = response["choices"][0]["message"]["content"]

        self.r.hset(
            self.config.REDIS_GPT_RESULT_KEY,
            task_id,
            json.dumps({"response": content})
        )

# Cell number 22

# server -> llama_server.py

import os
import json
import time
import requests
import traceback

# assumes these are already defined in earlier notebook cells:
# BaseServer, Config

class LLaMAServer(BaseServer):
    SERVER_KEY = "llama"

    def __init__(self, host="localhost", port=6379, db=0) -> None:
        super().__init__(host=host, port=port, db=db)
        self.config = Config()
        self.url = os.environ.get("LLaMA_SERVER_URL")

        self.headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {os.environ.get('LLaMA_API_KEY')}"
        }

    def get_task(self):
        data = self.r.rpop(self.config.REDIS_LLAMA_TASK_KEY)
        if not data:
            return False, None
        return True, json.loads(data)

    def on_error(self, e: Exception, data):
        self.config.log_error(self.SERVER_KEY, "".join(traceback.format_exception(e)))
        raise e

    def execute(self, data: dict):
        task_id = data.get("task_id")

        post_data = data.copy()
        post_data.pop("task_id", None)

        post_data["messages"] = json.dumps(post_data["messages"])
        post_data["model"] = "llama"
        post_data["password"] = "BioMedAgent"

        response = requests.post(
            f"{self.url}/openai",
            json=post_data,
            headers={"Content-Type": "application/json", "charset": "utf-8"}
        )

        resp_data = response.json()
        print(resp_data)

        uid = resp_data["uid"]

        while True:
            response = requests.get(f"{self.url}/openai/{uid}")
            print(response.text)

            if response.status_code != 200:
                time.sleep(1)
                continue

            resp_data = response.json()
            if not resp_data["result"]:
                time.sleep(1)
                continue

            text = resp_data["text"]
            self.r.hset(
                self.config.REDIS_LLAMA_RESULT_KEY,
                task_id,
                json.dumps({"response": text})
            )
            break

# Cell number 23

# server -> kb_retriever.py

import os
import json
import bisect
from BCEmbedding import RerankerModel

# assumes these are already defined in earlier notebook cells:
# BaseServer, Config

class KBRetriever(BaseServer):
    SERVER_KEY = "kbr"

    def __init__(self, host="localhost", port=6379, db=0, model_path=None) -> None:
        super().__init__(host=host, port=port, db=db)
        self.config = Config()

        if model_path is None:
            model_path = os.path.join(self.config.BASE_DIR, "model", "bce-reranker-base_v1")

        self.reranker_model = RerankerModel(model_name_or_path=model_path)

    def get_task(self):
        data = self.r.rpop(self.config.REDIS_MEMORY_TASK_POST_KEY)
        if not data:
            return False, None
        return True, json.loads(data)

    def execute(self, data: dict):
        task_id = data["task_id"]
        query_text = data["query_text"]
        query_item = data["query_item"]
        k = data.get("k", 5)
        threshold = data.get("threshold", 0.4)

        query_info = self.r.lrange(
            f"{self.config.REDIS_MEMORY_STORAGE_KEY}:{query_item}", 0, -1
        )

        query_list = []
        query_dict = {}

        for item in query_info:
            item = json.loads(item)
            query_list.append(item["key"])
            query_dict[item["key"]] = item["value"]

        if not query_list:
            self.r.hset(self.config.REDIS_MEMORY_TASK_GET_KEY, task_id, json.dumps([]))
            return

        rerank_results = self.reranker_model.rerank(query_text, query_list)

        # keep the original threshold logic
        scores = rerank_results["rerank_scores"]
        passages = rerank_results["rerank_passages"]

        k = min(
            k,
            bisect.bisect_left(scores, -threshold, key=lambda x: -x)
        )

        top_k_item = passages[:k]

        result = []
        for item in top_k_item:
            result.append(query_dict[item])

        self.r.hset(self.config.REDIS_MEMORY_TASK_GET_KEY, task_id, json.dumps(result))


# Cell number 24

# server -> memory_server.py

import os
import json
import bisect
from BCEmbedding import RerankerModel

# assumes these are already defined in earlier notebook cells:
# BaseServer, Config

class MemoryServer(BaseServer):
    SERVER_KEY = "memory"

    def __init__(self, host="localhost", port=6379, db=0, model_path=None) -> None:
        super().__init__(host=host, port=port, db=db)
        self.config = Config()

        if model_path is None:
            model_path = os.path.join(self.config.BASE_DIR, "model", "bce-reranker-base_v1")

        self.reranker_model = RerankerModel(model_name_or_path=model_path)

    def get_task(self):
        data = self.r.rpop(self.config.REDIS_MEMORY_TASK_KEY)
        if not data:
            return False, None
        return True, json.loads(data)

    def execute(self, data: dict):
        task_id = data.get("task_id")
        item_key = data.get("item_key")
        query_key = data.get("query_key")
        k = data.get("k")
        threshold = data.get("threshold", 0.4)

        memory = self.r.lrange(f"{self.config.REDIS_MEMORY_STORAGE_KEY}:{item_key}", 0, -1)

        info = {}
        query_list = []

        for item in memory:
            item = json.loads(item)
            if item["value"] is None:
                self.config.log_error(self.SERVER_KEY, json.dumps(data, indent=4))
                continue

            info[item["key"]] = {
                "value": item["value"],
                "id": item["id"],
                "question_id": item["question_id"],
                "key": item["key"],
            }
            query_list.append(item["key"])

        if not query_list:
            self.r.hset(self.config.REDIS_MEMORY_RESULT_KEY, task_id, json.dumps([]))
            return

        try:
            rerank_results = self.reranker_model.rerank(query_key, query_list)
        except AssertionError:
            self.r.hset(self.config.REDIS_MEMORY_RESULT_KEY, task_id, json.dumps([]))
            return

        scores = rerank_results["rerank_scores"]
        passages = rerank_results["rerank_passages"]

        k = min(
            k,
            bisect.bisect_left(scores, -threshold, key=lambda x: -x)
        )

        top_k_instances = passages[:k]

        result = []
        for item, score in zip(top_k_instances, scores):
            info[item]["score"] = score
            result.append(info[item])

        self.r.hset(self.config.REDIS_MEMORY_RESULT_KEY, task_id, json.dumps(result))



# Cell number 25

# Notebook / Colab compatibility patch for asyncio.run(...)
!pip -q install nest_asyncio

import asyncio
import nest_asyncio

nest_asyncio.apply()

def _notebook_safe_asyncio_run(coro):
    try:
        loop = asyncio.get_event_loop()
    except RuntimeError:
        loop = asyncio.new_event_loop()
        asyncio.set_event_loop(loop)

    return loop.run_until_complete(coro)

# monkey-patch asyncio.run so the copied BioMedAgent code works in notebooks
asyncio.run = _notebook_safe_asyncio_run

print("Patched asyncio.run for notebook-safe execution.")

# Cell number 26

import os
import subprocess
from pathlib import Path

# ------------------------------------------------------------
# 0) dependencies
# ------------------------------------------------------------
subprocess.run(
    ["pip", "-q", "install", "pandas", "scipy"],
    check=True
)

CODE_DIR = Path("/content/BioMedAgent/tool/code")
DOC_DIR = Path("/content/BioMedAgent/tool/doc")
CODE_DIR.mkdir(parents=True, exist_ok=True)
DOC_DIR.mkdir(parents=True, exist_ok=True)

t_test_code = r'''
import pandas as pd
from scipy.stats import ttest_ind

def _read_data_file(path):
    return pd.read_csv(path, sep="\t", index_col=0)

def _read_group_file(path):
    return pd.read_csv(path, sep="\t")

def t_test(data_file, group_file, groupA_name, groupB_name, id, group):
    data_df = _read_data_file(data_file)
    group_df = _read_group_file(group_file)

    data_df.columns = data_df.columns.map(str)
    group_df[id] = group_df[id].map(str)
    group_df[group] = group_df[group].map(str)

    groupA_samples = group_df.loc[group_df[group] == str(groupA_name), id].tolist()
    groupB_samples = group_df.loc[group_df[group] == str(groupB_name), id].tolist()

    groupA_samples = [s for s in groupA_samples if s in data_df.columns]
    groupB_samples = [s for s in groupB_samples if s in data_df.columns]

    if not groupA_samples:
        raise ValueError(f"No samples found for groupA={groupA_name!r}")
    if not groupB_samples:
        raise ValueError(f"No samples found for groupB={groupB_name!r}")

    rows = []
    for biomarker, row in data_df.iterrows():
        a = pd.to_numeric(row[groupA_samples], errors="coerce").dropna().astype(float)
        b = pd.to_numeric(row[groupB_samples], errors="coerce").dropna().astype(float)

        if len(a) >= 2 and len(b) >= 2:
            t_stat, p_value = ttest_ind(a, b, equal_var=True, nan_policy="omit")
        else:
            t_stat, p_value = float("nan"), float("nan")

        rows.append({
            "biomarker": biomarker,
            "feature": biomarker,
            "groupA": str(groupA_name),
            "groupB": str(groupB_name),
            "groupA_n": int(len(a)),
            "groupB_n": int(len(b)),
            "groupA_mean": float(a.mean()) if len(a) else float("nan"),
            "groupB_mean": float(b.mean()) if len(b) else float("nan"),
            "groupA_std": float(a.std(ddof=1)) if len(a) > 1 else float("nan"),
            "groupB_std": float(b.std(ddof=1)) if len(b) > 1 else float("nan"),
            "t_stat": float(t_stat) if pd.notna(t_stat) else float("nan"),
            "p_value": float(p_value) if pd.notna(p_value) else float("nan"),
        })

    out_df = pd.DataFrame(rows).sort_values(
        by=["p_value", "biomarker"], na_position="last"
    )

    out_file = "out_t_test.tsv"
    out_df.to_csv(out_file, sep="\t", index=False)
    return out_file
'''
(CODE_DIR / "t_test").write_text(t_test_code, encoding="utf-8")

# keep / create the matching doc file
t_test_doc = """# tool_name

t_test

# description

t_test performs an independent-samples t-test between two groups using a biomarker table and a grouping table.

# input

data_file
    Type: file
    Description: A table whose columns are sample names and whose rows are biomarker names.

group_file
    Type: file
    Description: A table containing sample IDs and grouping information.

groupA_name
    Type: str
    Description: Name of group A.

groupB_name
    Type: str
    Description: Name of group B.

id
    Type: str
    Description: Column name in group_file containing sample IDs.

group
    Type: str
    Description: Column name in group_file containing group labels.

# output

out_tTest_file
    Type: file
    Description: TSV file with t-test results.
"""
(DOC_DIR / "t_test").write_text(t_test_doc, encoding="utf-8")

print("Patched:", CODE_DIR / "t_test")
print("Doc exists:", (DOC_DIR / "t_test").exists())
print("Code exists:", (CODE_DIR / "t_test").exists())

#@title 🦁 Install extra Python dependency for BioMedAgent survival_curve

import sys, subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "docker"
])

import docker
print("✅ Python docker module is installed:", docker.__version__)

BASE_DIR = /content/BioMedAgent
LOG_DIR  = /content/BioMedAgent/log
[]
<class 'list'>
Patched: /content/BioMedAgent/tool/code/t_test
Now outputs both 'biomarker' and 'feature'.
Patched asyncio.run for notebook-safe execution.
Patched: /content/BioMedAgent/tool/code/t_test
Doc exists: True
Code exists: True
✅ Python docker module is installed: 7.1.0


In [10]:
# @title 🦁 **BioMedAgent** — Starting **GPTServer** and **CodeExecutor** ⚙️ { display-mode: "form" }
# Cell number 27, first part

import os
import asyncio
import threading

from pathlib import Path
import requests
import os

# ------------------------------------------------------------------
# 0) start background servers only once
# ------------------------------------------------------------------
if "gpt_server_started" not in globals():
    gpt_server = GPTServer()
    threading.Thread(target=gpt_server.run, daemon=True).start()
    gpt_server_started = True
    print("GPTServer started.")
else:
    print("GPTServer already running.")

if "code_server_started" not in globals():
    code_server = CodeExecutor()
    threading.Thread(target=code_server.run, daemon=True).start()
    code_server_started = True
    print("CodeExecutor started.")
else:
    print("CodeExecutor already running.")


# ------------------------------------------------------------------
# 3) build question_info exactly like demo.py, but with Drive paths
# ------------------------------------------------------------------
base_info = {
    "machine_learning": {
        "question": "I have a dataset {heart_disease.csv}, the “Target” column is the target column and the other columns are the feature columns, please perform a singular value decomposition downscaling on this dataset.",
        "files": [{
            "name": "heart_disease.csv",
            "path": HEART_PATH
        }]
    },
    "statistics_t_test": {
        "question": "There are two existing data tables, {data1.tsv} is a biomarker data table, with sample names as column names and biomarker names as row names, {group1.tsv} is a data table containing sample grouping information, where the two groups are 1 and 2, with the column name ID as sample names and the column name group as grouping information. Please use the tool t_test to perform an independent sample t-test based on this information.",
        "files": [
            {"name": "data1.tsv", "path": DATA1_PATH},
            {"name": "group1.tsv", "path": GROUP1_PATH},
        ]
    },
    "statistics_qq_plot": {
        "question": "There is a dataset {boxplot.tsv}, Group2 column is the grouping information, which contains treat1 and treat2, please analyze whether the data in Value column of the two groupings of treat1 and treat2 have similar distribution characteristics, and plot QQ plot.",
        "files": [{
            "name": "boxplot.tsv",
            "path": BOXPLOT_PATH
        }]
    },
    "visualization_survival_plot": {
        "question": "There is a data table {TCGA_LIHC_survival.txt} that contains information related to patient survival. The column named OS.time represents the survival time of the patients, the column named OS indicates the survival event status of the patients, and the column named gender is a variable that affects survival. Please use the tool survival_curve to plot the survival curve based on this information.",
        "files": [{
            "name": "TCGA_LIHC_survival.txt",
            "path": SURVIVAL_PATH
        }]
    },
    "visualization_violin_plot": {
        "question": "There is a data file {plot.tsv}, the Sex column is the gender of the sample, Age_Group is the different age grouping corresponding to each gender, please use the violin plot to show the distribution of WBC for each age grouping of different genders.",
        "files": [{
            "name": "plot.tsv",
            "path": PLOT_PATH
        }]
    },
}
question_info = base_info[TASK_TYPE]

# ------------------------------------------------------------------
# 4) create config and verify runtime filesystem layout
# ------------------------------------------------------------------
config = Config(generate_task_id())

os.makedirs(os.path.join(config.TASK_DIR, config.get_task()), exist_ok=True)
os.makedirs(config.TOOL_DOC_DIR, exist_ok=True)
os.makedirs(config.TOOL_CODE_DIR, exist_ok=True)

BASE = "https://raw.githubusercontent.com/BOBQWERA/BioMedAgent/main"

tool_doc_dir = Path("/content/BioMedAgent/tool/doc")
tool_code_dir = Path("/content/BioMedAgent/tool/code")

tool_doc_dir.mkdir(parents=True, exist_ok=True)
tool_code_dir.mkdir(parents=True, exist_ok=True)

files = {
    tool_doc_dir / "survival_curve": f"{BASE}/tool/doc/survival_curve",
    tool_code_dir / "survival_curve": f"{BASE}/tool/code/survival_curve",
}

for out_path, url in files.items():
    print(f"Downloading {url}")
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    out_path.write_text(r.text, encoding="utf-8")
    print(f"✅ Saved: {out_path}")

print("\nCurrent tool/doc files:", sorted(os.listdir(tool_doc_dir)))
print("Current tool/code files:", sorted(os.listdir(tool_code_dir)))


print("TOOL_DOC_DIR :", config.TOOL_DOC_DIR)
print("TOOL_CODE_DIR:", config.TOOL_CODE_DIR)
print("doc exists?  ", os.path.exists(config.TOOL_DOC_DIR))
print("code exists? ", os.path.exists(config.TOOL_CODE_DIR))

# ------------------------------------------------------------------
# 4a-fixed) Patch survival_curve tool for Colab: no Docker required
# ------------------------------------------------------------------
from pathlib import Path

survival_code_path = Path(config.TOOL_CODE_DIR) / "survival_curve"

survival_code_path.write_text(r'''
def survival_curve(survival_file, time, event, variable):
    import os
    import pandas as pd
    import numpy as np

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    # Read tab-separated survival file first; fallback to comma-separated.
    try:
        df = pd.read_csv(survival_file, sep="\t")
    except Exception:
        df = pd.read_csv(survival_file)

    required_cols = [time, event, variable]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}. Available columns: {list(df.columns)}")

    df = df[required_cols].copy()
    df[time] = pd.to_numeric(df[time], errors="coerce")

    # Convert event status to 0/1 if needed.
    if not pd.api.types.is_numeric_dtype(df[event]):
        event_map = {
            "0": 0, "1": 1,
            "alive": 0, "dead": 1,
            "censored": 0, "event": 1,
            "false": 0, "true": 1,
            "no": 0, "yes": 1,
        }
        df[event] = (
            df[event]
            .astype(str)
            .str.strip()
            .str.lower()
            .map(event_map)
        )
    else:
        df[event] = pd.to_numeric(df[event], errors="coerce")

    df = df.dropna(subset=[time, event, variable])
    df[event] = df[event].astype(int)

    if df.empty:
        raise ValueError("No valid rows remain after filtering missing survival data.")

    def km_curve(sub):
        sub = sub.sort_values(time)
        t = sub[time].to_numpy(dtype=float)
        e = sub[event].to_numpy(dtype=int)

        event_times = np.sort(np.unique(t[e == 1]))

        xs = [0.0]
        ys = [1.0]
        surv = 1.0

        for tt in event_times:
            n_risk = np.sum(t >= tt)
            n_events = np.sum((t == tt) & (e == 1))

            if n_risk > 0:
                new_surv = surv * (1.0 - n_events / n_risk)

                xs.extend([tt, tt])
                ys.extend([surv, new_surv])

                surv = new_surv

        max_t = float(np.max(t))
        xs.append(max_t)
        ys.append(surv)

        return xs, ys

    plt.figure(figsize=(7, 5))

    for group_name, sub in df.groupby(variable):
        xs, ys = km_curve(sub)
        plt.step(xs, ys, where="post", label=f"{variable}={group_name} (n={len(sub)})")

    plt.xlabel(time)
    plt.ylabel("Survival probability")
    plt.title(f"Kaplan-Meier survival curve stratified by {variable}")
    plt.ylim(0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    out_file = "out_survival.pdf"
    plt.savefig(out_file, bbox_inches="tight")
    plt.close()

    return out_file
''', encoding="utf-8")

print(f"✅ Patched Colab-native survival_curve tool: {survival_code_path}")

doc_files = sorted([x for x in os.listdir(config.TOOL_DOC_DIR) if not x.startswith(".")])
code_files = sorted([x for x in os.listdir(config.TOOL_CODE_DIR) if not x.startswith(".")])

print("doc files :", doc_files)
print("code files:", code_files)

# ------------------------------------------------------------------
# 4b) require only the tools actually needed by the selected task
# ------------------------------------------------------------------
required_tools_by_task = {
    "statistics_t_test": ["t_test"],
    "statistics_qq_plot": [],
    "machine_learning": [],
    "visualization_survival_plot": ["survival_curve"],
    "visualization_violin_plot": [],
    "omics": ["cel2matrix"],
}

required_tools = required_tools_by_task.get(TASK_TYPE, [])
print("required tools for this task:", required_tools)

for required in required_tools:
    if required not in doc_files:
        raise FileNotFoundError(
            f"Missing tool doc file: {required} in {config.TOOL_DOC_DIR}"
        )
    if required not in code_files:
        raise FileNotFoundError(
            f"Missing tool code file: {required} in {config.TOOL_CODE_DIR}"
        )

# ------------------------------------------------------------------
# 5) create task
# ------------------------------------------------------------------
task = Task(question_info, config)

# ------------------------------------------------------------------
# 6) instantiate agents
# ------------------------------------------------------------------
linguist = Linguist(task)
translator = Translator(task)
prompt_engineer = PromptEngineer(task)
file_analyst = FileAnalyst(task)
tool_scorer = ToolScorer(task)
tool_descriptor = ToolDescriptor(task)
tool_rescorer = ToolReScorer(task)
workflow_designer = WorkflowDesigner(task)
tool_analyst = ToolAnasyst(task)
workflow_formatter = WorkflowFormatter(task)
action_designer = ActionDesigner(task)
programmer = Programmer(task)
summary_analyst = SummaryAnalyst(task)
workflow_redesigner = WorkflowReDesigner(task)

GPTServer already running.
CodeExecutor already running.
✅ Saved: /content/BioMedAgent/tool/doc/survival_curve
✅ Saved: /content/BioMedAgent/tool/code/survival_curve

Current tool/doc files: ['survival_curve', 't_test']
Current tool/code files: ['survival_curve', 't_test']
TOOL_DOC_DIR : /content/BioMedAgent/tool/doc
TOOL_CODE_DIR: /content/BioMedAgent/tool/code
doc exists?   True
code exists?  True
✅ Patched Colab-native survival_curve tool: /content/BioMedAgent/tool/code/survival_curve
doc files : ['survival_curve', 't_test']
code files: ['survival_curve', 't_test']
required tools for this task: ['survival_curve']
/content/BioMedAgent_UserData/TCGA_LIHC_survival.txt


In [11]:
# @title 🦁 Launching BioMedAgent 🚀 { display-mode: "form" }
# Cell 27, part 2
# ------------------------------------------------------------------
# 7) run the same pipeline as demo.py
# ------------------------------------------------------------------
linguist.check_English_task()
translator.translate_question()
prompt_engineer.refine_prompt()
asyncio.run(file_analyst.analyse_files())
asyncio.run(tool_scorer.tools_score())
asyncio.run(tool_descriptor.tools_describe())
asyncio.run(tool_rescorer.tools_score())
asyncio.run(tool_descriptor.tools_describe())
workflow_designer.workflow_design()
asyncio.run(tool_analyst.tools_analyse())
workflow_formatter.workflow_format()
asyncio.run(action_designer.actions_design())

ok, suggestion = programmer.programming()

retry_times = 1
while not ok and retry_times < 3:
    retry_times += 1
    task.backtrack_update()
    workflow_redesigner.workflow_redesign(suggestion)
    asyncio.run(tool_analyst.tools_analyse())
    workflow_formatter.workflow_format()
    asyncio.run(action_designer.actions_design())
    task.status.status_update("workflow")
    ok, suggestion = programmer.programming()

if not ok:
    raise Exception("Task failed")

summary_analyst.summary()

print("\nTask finished.")
print("Task folder:", config.task_path)

<RESULT>YES</RESULT>
<PROMPT> 
1. **Task**: Plot a survival curve using patient survival data.
2. **Data File**: TCGA_LIHC_survival.txt
3. **Key Columns**:
   - **OS.time**: Represents the survival time of the patients.
   - **OS**: Indicates the survival event status of the patients (e.g., alive or deceased).
   - **gender**: A variable that may influence survival outcomes.
4. **Tool**: Use the tool named `survival_curve` for plotting.
5. **Requirements**:
   - Ensure that the data is pre-processed to handle any missing values.
   - Consider stratifying the survival curve by gender to analyze differences in survival rates.
   - Label axes and provide a legend for clarity in the plot.
6. **Output**: A visual representation of the survival curve with appropriate annotations.
</PROMPT>
<ANALYSE> 
The file TCGA_LIHC_survival.txt is a text file that likely contains tabular data related to patient survival in liver cancer (LIHC) from the TCGA (The Cancer Genome Atlas) project. The key colum